# PQID — GitHub Acquisition and Benchmark-Preparation Notebook

This notebook documents the GitHub-based acquisition workflow used to construct the 2026 rebuilt version of PQID. It combines circuit retrieval, append-only recall expansion, raw-pool enrichment, extraction-quality auditing, master-corpus construction, and late-stage benchmark/release filtering within a single reproducible record.

The notebook is organized into four substantive stages. The first stage constructs the baseline raw pool through curated repositories, targeted GitHub code search, organization-level repository enumeration, and topic-based repository discovery. The second stage performs a broader but still controlled recall expansion through an append-only aggressive rescrape. The third stage executes a final high-yield recall expansion, followed by the post-processing steps required to obtain corrected broad-pool counts. The fourth stage builds a master processable corpus for all downstream instruction-generation work while deferring benchmark, balanced, and public-release subset construction to explicit late-stage audit and filtering cells.

At the time of writing, the corrected outputs associated with this notebook are as follows:

- merged raw acquisition pool: `91,719` circuits
- validated materialized circuits: `14,267`
- validated circuits with `gate_count > 0`: `13,530`
- default master processable corpus target: `13,530`
- audit-only strict benchmark view before mutation cleaning: `803`
- audit-only extended benchmark view before mutation cleaning: `11,999`
- cleaned strict benchmark view after mutation-path exclusion: `415`
- cleaned extended benchmark view after mutation-path exclusion: `734`

The notebook is intentionally resume-aware during the acquisition stages. Processed URLs and seen circuit hashes are persisted to disk so that interrupted runs may be resumed without invalidating previously completed work. From the master-corpus stage onward, later benchmark and release subsets are treated as explicit end-stage derivations so that reviewers can reconstruct the same filtering decisions from the same fully processed corpus.


## Cell 1 — Imports and Global Configuration

This cell establishes the global runtime context for the notebook. It imports the standard Python libraries required throughout the acquisition and merge workflow, defines repository-relative paths, and initializes shared constants referenced by subsequent cells. A small display helper is also introduced so that notebook outputs remain publication-safe even when executed on a local machine.

The purpose of this cell is primarily infrastructural. Many later cells assume that file locations, schema field templates, and shared runtime constants already exist in memory. Executing this cell at the beginning of a fresh kernel session therefore ensures that subsequent acquisition and post-processing steps operate against a consistent configuration state.

**Principal variables and helper introduced in the code cell**

- `PQID_ROOT`: repository-local root for the PQID package
- `REPO_ROOT`: parent repository root used to resolve shared files such as `github_urls.txt`
- `BASE`: processed-data directory used by acquisition outputs
- `GITHUB_URLS_FILE`: curated repository list used by the baseline scrape
- `OUTPUT_FILE` and `PROCESSED_FILE`: baseline raw output and processed-URL cache
- `API_BASE`, `SCRAPE_DATE`, `CORE_SLEEP`, `SEARCH_SLEEP`: API and rate-limit control constants
- `display_path()`: helper used to present repository-relative paths in notebook output


In [ ]:
import base64
import hashlib
import json
import os
import re
import time
from datetime import date
from pathlib import Path

import requests

# ---------------------------------------------------------------------------
# Path auto-detection — no hardcoded user directories
# ---------------------------------------------------------------------------
# VS Code sets __vsc_ipynb_file__; JupyterLab/classic fall back to cwd.
try:
    _NB_DIR = Path(__vsc_ipynb_file__).resolve().parent   # VS Code
except NameError:
    _NB_DIR = Path().resolve()                             # JupyterLab / classic

# Expected repo layout:
#   <repo-root>/
#       github_urls.txt
#       PQID/
#           scripts/          ← this notebook lives here
#           data/processed/   ← output directory
PQID_ROOT        = _NB_DIR.parent
REPO_ROOT        = PQID_ROOT.parent
BASE             = PQID_ROOT / "data" / "processed"
GITHUB_URLS_FILE = REPO_ROOT / "github_urls.txt"

def display_path(path):
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT))
    except Exception:
        try:
            return str(Path(path))
        except Exception:
            return str(path)

# ---------------------------------------------------------------------------
# Secret resolution — never hardcoded
# ---------------------------------------------------------------------------
# Priority (checked in helper functions below):
#   1. direct environment variable (e.g. GITHUB_TOKEN, OPENAI_API_KEY)
#   2. file override environment variable (e.g. GITHUB_TOKEN_FILE)
#   3. repo-local untracked file
#   4. user-level external secret file in ~/.pqid_secrets/
USER_SECRET_DIR = (
    Path(os.environ["PQID_SECRET_DIR"]).expanduser()
    if os.environ.get("PQID_SECRET_DIR", "").strip()
    else Path.home() / ".pqid_secrets"
)

GITHUB_TOKEN_CANDIDATES = [
    REPO_ROOT / ".github_token",
    PQID_ROOT / ".github_token",
    USER_SECRET_DIR / "github_token.txt",
]
OPENAI_API_KEY_CANDIDATES = [
    REPO_ROOT / ".openai_api_key",
    PQID_ROOT / ".openai_api_key",
    USER_SECRET_DIR / "openai_api_key.txt",
]

OPENAI_NAMED_SECRET_FILENAMES = [
    "OPENAI_API_KEY_PQID_GPT54_V2.txt",
    "OPENAI_API_KEY_PQID_GPT54_V1.txt",
    "OPENAI_API_KEY_PQID_V1.txt",
]
NAMED_SECRET_FILENAMES = {
    "GITHUB_TOKEN_FILE": "GITHUB_TOKEN_PQID_V1.txt",
}
NAMED_SECRET_SEARCH_ROOTS = [
    USER_SECRET_DIR,
    Path.home() / "Desktop",
    Path.home() / "Documents",
    Path.home() / "Downloads",
    Path.home() / "AppData" / "Roaming",
    Path.home() / "AppData" / "Local",
]

OUTPUT_FILE    = BASE / "circuits_unified.jsonl"
PROCESSED_FILE = BASE / "circuits_unified_processed.txt"

API_BASE    = "https://api.github.com"
SCRAPE_DATE = str(date.today())

# Rate limits (authenticated): Core=5000/hr, Search=30/min
CORE_SLEEP   = 0.25   # 4 req/s
SEARCH_SLEEP = 2.5    # 24 req/min

MAX_FILE_SIZE_BYTES = 300_000
MIN_CIRCUIT_TOKENS  = 3


# ---------------------------------------------------------------------------
# Query groups
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Search queries (targeted Qiskit patterns divided in core 25 queries + extened queries)
# ---------------------------------------------------------------------------
SEARCH_QUERIES_CORE = [
    "QuantumCircuit( language:python",
    "QuantumCircuit( language:jupyter-notebook",
    "from qiskit import QuantumCircuit language:python",
    "from qiskit.circuit import QuantumCircuit language:python",
    "qiskit.circuit.QuantumCircuit language:python",
    "QuantumCircuit qc.h qc.cx language:python",
    "QuantumCircuit qc.measure language:python",
    "QuantumCircuit transpile language:python",
    "QuantumCircuit ParameterVector language:python",
    "QuantumCircuit qc.barrier language:python",
    "QuantumCircuit num_qubits language:python",
    "QuantumCircuit qreg creg language:python",
    "QuantumCircuit qc.append language:python",
    "QuantumCircuit qc.compose language:python",
    "qiskit variational quantum circuit language:python",
    "qiskit VQE ansatz language:python",
    "qiskit QAOA language:python",
    "qiskit QFT QuantumCircuit language:python",
    "qiskit grover language:python",
    "qiskit teleportation circuit language:python",
    "qiskit error correction language:python",
    "qiskit amplitude estimation language:python",
    "qiskit phase estimation language:python",
    "qiskit swap test language:python",
    "qiskit GHZ state language:python",
    "qiskit bell state language:python",
]

SEARCH_QUERIES_EXTENDED = [
    "from qiskit.primitives import Sampler language:python",
    "from qiskit.primitives import Estimator language:python",
    "Sampler() qiskit language:python",
    "Estimator() qiskit language:python",
    "from qiskit_ibm_runtime import Sampler language:python",
    "from qiskit_ibm_runtime import Estimator language:python",
    "from qiskit_aer import AerSimulator language:python",
    "AerSimulator language:python",
    "Statevector.from_instruction qiskit language:python",
    "from qiskit.quantum_info import Statevector language:python",
    "from qiskit import transpile language:python",
    "PassManager qiskit language:python",
    "from qiskit.transpiler import PassManager language:python",
    "CouplingMap qiskit language:python",
    "Layout qiskit language:python",
    "generate_preset_pass_manager qiskit language:python",
    "from qiskit.circuit.library import TwoLocal language:python",
    "from qiskit.circuit.library import RealAmplitudes language:python",
    "from qiskit.circuit.library import EfficientSU2 language:python",
    "from qiskit.circuit.library import ZZFeatureMap language:python",
    "from qiskit.circuit.library import PauliFeatureMap language:python",
    "from qiskit_algorithms import VQE language:python",
    "from qiskit_algorithms import QAOA language:python",
    "from qiskit_algorithms import Grover language:python",
    "from qiskit_algorithms import PhaseEstimation language:python",
    "from qiskit_algorithms import AmplitudeEstimation language:python",
    "qiskit_machine_learning language:python",
    "qiskit_optimization language:python",
    "VQC qiskit language:python",
    "quantum kernel qiskit language:python",
    "qc.draw() qiskit language:jupyter-notebook",
    "plot_histogram qiskit language:jupyter-notebook",
    "measure_all() qiskit language:jupyter-notebook",
    "import qiskit as qk QuantumCircuit language:python",
    "import qiskit as qk language:python",
    "qc = QuantumCircuit( language:python",
    "circ = QuantumCircuit( language:python",
]

SEARCH_QUERIES = list(dict.fromkeys(SEARCH_QUERIES_CORE + SEARCH_QUERIES_EXTENDED))

# Orgs to fully enumerate
ORGS = ["Qiskit", "qiskit-community"]

# Topics to enumerate repos (9 core topics + exteneded list of topics)
TOPICS_CORE = [
    "qiskit", "quantum-computing", "quantum-circuit",
    "quantum-machine-learning", "variational-quantum-eigensolver",
    "qaoa", "quantum-algorithms", "qasm", "quantum-simulation",
]

TOPICS_EXTENDED = [
    "ibm-quantum",
    "qiskit-aer",
    "quantum-programming",
    "quantum-education",
    "nisk",
    "quantum-error-correction",
    "quantum-ml",
    "quantum-neural-network",
    "quantum-kernel",
    "vqe",
    "vqc",
    "grover",
    "phase-estimation",
    "quantum-optimization",
    "quantum-circuit-simulation",
]

TOPICS = list(dict.fromkeys(TOPICS_CORE + TOPICS_EXTENDED))

# ---------------------------------------------------------------------------

BASE.mkdir(parents=True, exist_ok=True)
print(f"Notebook dir : {display_path(_NB_DIR)}")
print(f"PQID root    : {display_path(PQID_ROOT)}")
print(f"Repo root    : {display_path(REPO_ROOT)}")
print(f"Output       : {display_path(OUTPUT_FILE)}")
print(f"GitHub token : {"available" if any(p.is_file() for p in GITHUB_TOKEN_CANDIDATES) or os.environ.get("GITHUB_TOKEN", "").strip() or os.environ.get("GITHUB_TOKEN_FILE", "").strip() else "not found"}")
print(f"OpenAI key   : {"available" if any(p.is_file() for p in OPENAI_API_KEY_CANDIDATES) or os.environ.get("OPENAI_API_KEY", "").strip() or os.environ.get("OPENAI_API_KEY_FILE", "").strip() else "not found"}")
print(f"URLs file    : {display_path(GITHUB_URLS_FILE)}  {"available" if GITHUB_URLS_FILE.exists() else "missing"}")
print(f"Date         : {SCRAPE_DATE}")
print(f"Queries      : {len(SEARCH_QUERIES)}")


## Cell 1A — Local Secret Auto-Discovery

This local-only setup cell searches common user directories for the configured OpenAI and GitHub secret filenames used on this machine, including `OPENAI_API_KEY_PQID_GPT54_V2.txt`, `OPENAI_API_KEY_PQID_GPT54_V1.txt`, `OPENAI_API_KEY_PQID_V1.txt`, and `GITHUB_TOKEN_PQID_V1.txt`. If they are found, the cell sets `OPENAI_API_KEY_FILE` and `GITHUB_TOKEN_FILE` inside the current kernel session.

Run this cell once in every fresh kernel session before any API-dependent cells. No absolute paths are written into the notebook source; only the stable filenames are referenced.


In [ ]:
AUTO_OPENAI_SECRET_FILENAMES = [
    "OPENAI_API_KEY_PQID_GPT54_V2.txt",
    "OPENAI_API_KEY_PQID_GPT54_V1.txt",
    "OPENAI_API_KEY_PQID_V1.txt",
]
AUTO_SECRET_FILENAMES = {
    "GITHUB_TOKEN_FILE": "GITHUB_TOKEN_PQID_V1.txt",
}

AUTO_SECRET_SEARCH_ROOTS = [
    USER_SECRET_DIR,
    Path.home() / "Desktop",
    Path.home() / "Documents",
    Path.home() / "Downloads",
    Path.home() / "AppData" / "Roaming",
    Path.home() / "AppData" / "Local",
]

def find_named_secret(filename):
    for root in AUTO_SECRET_SEARCH_ROOTS:
        if not root.exists():
            continue
        try:
            for candidate in root.rglob(filename):
                if candidate.is_file():
                    return candidate
        except (OSError, PermissionError):
            continue
    return None

configured = {}
if os.environ.get("OPENAI_API_KEY_FILE", "").strip():
    configured["OPENAI_API_KEY_FILE"] = "already configured"
else:
    found = None
    matched_name = None
    for filename in AUTO_OPENAI_SECRET_FILENAMES:
        found = find_named_secret(filename)
        if found is not None:
            matched_name = filename
            break
    if found is not None:
        os.environ["OPENAI_API_KEY_FILE"] = str(found)
        configured["OPENAI_API_KEY_FILE"] = f"configured from {matched_name}"
    else:
        configured["OPENAI_API_KEY_FILE"] = f"not found: {', '.join(AUTO_OPENAI_SECRET_FILENAMES)}"

for env_name, filename in AUTO_SECRET_FILENAMES.items():
    if os.environ.get(env_name, "").strip():
        configured[env_name] = "already configured"
        continue
    found = find_named_secret(filename)
    if found is not None:
        os.environ[env_name] = str(found)
        configured[env_name] = f"configured from {filename}"
    else:
        configured[env_name] = f"not found: {filename}"

print("OPENAI_API_KEY_FILE:", configured["OPENAI_API_KEY_FILE"])
print("GITHUB_TOKEN_FILE:", configured["GITHUB_TOKEN_FILE"])


## Cell 2 — Input/Output and API Helper Functions

This cell defines the low-level helper functions that support file persistence and GitHub API access. These helpers handle authenticated requests, incremental JSONL writes, and the loading and saving of resume-state artifacts such as processed URL sets and seen-hash caches.

Methodologically, this cell provides the notebook with a persistent execution substrate. Rather than treating each strategy cell as a self-contained script, the notebook centralizes file and API behavior here so that later strategies can reuse the same retry logic, output format, and resume semantics.

**Principal functions defined in the code cell**

- `load_token()`: loads the GitHub token from a local file or environment variable
- `make_session()`: creates the authenticated GitHub session
- `api_get()`: performs API requests with backoff-aware rate-limit handling
- `load_processed()` and `mark_processed()`: persist and restore processed URL state
- `load_seen_hashes()`: reconstructs the set of already-seen circuit hashes from an output file
- `append_circuit()`: appends extracted entries to the JSONL output incrementally


In [ ]:
def find_named_secret_file(filename: str) -> Path | None:
    for root in NAMED_SECRET_SEARCH_ROOTS:
        if not root.exists():
            continue
        try:
            for candidate in root.rglob(filename):
                if candidate.is_file():
                    return candidate
        except (OSError, PermissionError):
            continue
    return None


def load_secret_value(
    value_env_name: str,
    file_env_name: str,
    file_candidates: list[Path],
    named_filenames: list[str] | None = None,
) -> str:
    value = os.environ.get(value_env_name, "").strip()
    if value:
        return value

    override = os.environ.get(file_env_name, "").strip()
    if override:
        candidate = Path(override).expanduser()
        if candidate.is_file():
            return candidate.read_text(encoding="utf-8").strip()

    for candidate in file_candidates:
        if candidate.is_file():
            return candidate.read_text(encoding="utf-8").strip()

    for filename in named_filenames or []:
        discovered = find_named_secret_file(filename)
        if discovered is not None:
            return discovered.read_text(encoding="utf-8").strip()

    return ""


def load_token() -> str:
    token = load_secret_value(
        "GITHUB_TOKEN",
        "GITHUB_TOKEN_FILE",
        GITHUB_TOKEN_CANDIDATES,
        [NAMED_SECRET_FILENAMES["GITHUB_TOKEN_FILE"]],
    )
    if not token:
        raise SystemExit(
            "ERROR: GitHub token not found. Set GITHUB_TOKEN, set GITHUB_TOKEN_FILE, or create ~/.pqid_secrets/github_token.txt"
        )
    return token


def make_session(token: str) -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
    })
    return s


def api_get(session, url: str, params: dict = None, sleep: float = CORE_SLEEP):
    """GET with rate-limit backoff. Returns parsed JSON or None."""
    for attempt in range(5):
        try:
            resp = session.get(url, params=params, timeout=30)
            if resp.status_code == 200:
                time.sleep(sleep)
                return resp.json()
            if resp.status_code in (403, 429):
                reset = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
                wait  = max(1, reset - int(time.time())) + 5
                print(f"  Rate limit — waiting {wait}s ...", flush=True)
                time.sleep(wait)
                continue
            if resp.status_code == 422:
                return None   # Search API: unprocessable (bad query / too many results)
            time.sleep(sleep)
            return None
        except Exception as e:
            time.sleep(2 ** attempt)
    return None


def load_processed(path: Path) -> set:
    if not path.exists():
        return set()
    with open(path, encoding="utf-8") as f:
        return {l.strip() for l in f if l.strip()}


def mark_processed(url: str, path: Path) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(url + "\n")


def load_seen_hashes(path: Path) -> set:
    if not path.exists():
        return set()
    seen = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                e = json.loads(line)
                ch = e.get("metadata", {}).get("circuit_hash", "")
                if ch:
                    seen.add(ch)
            except Exception:
                pass
    return seen


def append_circuit(entry: dict, path: Path) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


print("I/O and API helpers defined.")

## Cell 3 — Circuit Extraction Logic for Python and Notebook Sources

This cell defines the extraction functions used to identify candidate quantum-circuit snippets from raw GitHub file content. It includes logic for both Python source files and Jupyter notebook code cells, together with more permissive extraction paths suited to educational and experimental repositories.

This stage is analytically important because extraction quality strongly conditions downstream validation outcomes. The enrichment, audit, and benchmark-tiering stages all inherit the consequences of the snippet-selection logic defined here. For that reason, the extraction functions are established centrally and reused throughout the acquisition workflow rather than redefined independently in each strategy cell.

**Principal functions defined in the code cell**

- `_extract_function_blocks()`: extracts function-scoped candidate circuit blocks
- `_extract_module_level_blocks()`: extracts module-level candidate circuit blocks
- `extract_circuits_python()`: applies the extraction logic to Python source files
- `extract_circuits_notebook()`: applies the extraction logic to notebook code-cell content


In [ ]:
def _extract_function_blocks(lines: list) -> list:
    """
    Extract complete function definitions containing QuantumCircuit construction.
    Returns list of (code, start_line, end_line) — 1-indexed, GitHub anchor convention.
    """
    blocks = []
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        if re.match(r"^def\s+\w+\s*\(", stripped):
            start_idx = i
            indent = len(line) - len(line.lstrip())
            body_lines = [line]
            j = i + 1
            while j < len(lines):
                next_line = lines[j]
                if next_line.strip() == "":
                    body_lines.append(next_line)
                    j += 1
                    continue
                next_indent = len(next_line) - len(next_line.lstrip())
                if next_indent > indent:
                    body_lines.append(next_line)
                    j += 1
                else:
                    break
            body = "\n".join(body_lines).rstrip()
            if "QuantumCircuit(" in body and len(body.split()) >= MIN_CIRCUIT_TOKENS:
                blocks.append((body, start_idx + 1, j))
            i = j
        else:
            i += 1
    return blocks


def _extract_module_level_blocks(lines: list) -> list:
    """
    Extract module-level QuantumCircuit construction blocks.
    Returns list of (code, start_line, end_line) — 1-indexed.
    """
    blocks = []
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        if (not stripped or stripped.startswith("#")
                or stripped.startswith("import ")
                or stripped.startswith("from ")
                or re.match(r"^(class|def)\s", stripped)):
            i += 1
            continue
        if "QuantumCircuit(" in stripped and not line.startswith(" "):
            start_idx = i
            block_lines = [line]
            j = i + 1
            while j < len(lines):
                nl = lines[j]
                ns = nl.strip()
                if not ns or ns.startswith("#"):
                    j += 1
                    block_lines.append(nl)
                    continue
                if nl.startswith(" ") or nl.startswith("\t"):
                    block_lines.append(nl)
                    j += 1
                elif ns and not nl[0].isspace():
                    if any(tok in ns for tok in (
                        ".h(", ".cx(", ".ccx(", ".measure", ".barrier",
                        ".ry(", ".rz(", ".rx(", ".x(", ".y(", ".z(", ".s(",
                        ".t(", ".p(", ".u(", ".swap(", ".cz(", ".ch(", ".cp(",
                        ".append(", ".compose(", ".draw(", ".transpile(",
                        "qc.", "circuit.", "qreg", "creg", "QuantumRegister(",
                        "ClassicalRegister(", "ParameterVector(",
                    )):
                        block_lines.append(nl)
                        j += 1
                    else:
                        break
                else:
                    break
            block = "\n".join(block_lines).rstrip()
            if len(block.split()) >= MIN_CIRCUIT_TOKENS:
                blocks.append((block, start_idx + 1, j))
            i = j
        else:
            i += 1
    return blocks


def extract_circuits_python(code: str) -> list:
    """Extract QuantumCircuit blocks from Python source. Returns (code, start, end) tuples."""
    if "QuantumCircuit" not in code:
        return []
    lines = code.splitlines()
    results = []
    func_codes = set()
    for tup in _extract_function_blocks(lines):
        results.append(tup)
        func_codes.add(tup[0])
    for tup in _extract_module_level_blocks(lines):
        if tup[0] not in func_codes:
            results.append(tup)
    return results


def extract_circuits_notebook(raw_json: str) -> list:
    """Extract QuantumCircuit blocks from each code cell of a Jupyter notebook."""
    try:
        nb = json.loads(raw_json)
    except Exception:
        return []
    results = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)
        for (blk, _sl, _el) in extract_circuits_python(src):
            results.append((blk, None, None))
    return results


print("Extraction functions defined.")

## Cell 4 — Repository Traversal and File-Fetch Helpers

This cell defines the helper functions responsible for repository traversal and file retrieval through the GitHub API. It resolves default branches, enumerates repository contents, fetches raw file payloads, and normalizes file-level metadata into a form suitable for the extraction functions introduced above.

Conceptually, Cells 2–4 together form the operational core of the baseline acquisition pipeline: API access, content retrieval, and circuit extraction. All subsequent retrieval strategies rely on this shared substrate.

**Principal functions defined in the code cell**

- `get_default_branch()`: resolves the repository’s default branch through the GitHub API
- `get_repo_py_files()`: enumerates candidate Python and notebook files in a repository
- `fetch_file_circuits()`: downloads a file and extracts circuit entries from it
- `process_repo()`: orchestrates repository-level traversal and extraction


In [ ]:
def get_default_branch(session, owner: str, repo: str) -> str:
    data = api_get(session, f"{API_BASE}/repos/{owner}/{repo}")
    if data:
        return data.get("default_branch", "main")
    return "main"


def get_repo_py_files(session, owner: str, repo: str, branch: str) -> list:
    """Return all .py and .ipynb file paths in repo using the git tree API."""
    tree_url = f"{API_BASE}/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    data = api_get(session, tree_url)
    if not data:
        return []
    tree = data.get("tree", [])
    return [
        item["path"] for item in tree
        if item["type"] == "blob"
        and item["path"].lower().endswith((".py", ".ipynb"))
        and item.get("size", 0) <= MAX_FILE_SIZE_BYTES
        and not any(seg in item["path"] for seg in (
            "test", "__pycache__", ".egg-info", "node_modules",
            "docs/", "build/", "dist/", ".tox/",
        ))
    ]


def fetch_file_circuits(
    session, owner: str, repo: str, path: str, branch: str,
    source_tag: str, seen_hashes: set, processed: set,
) -> list:
    """Download one file, extract circuits, return new entry dicts."""
    file_url = f"https://github.com/{owner}/{repo}/blob/{branch}/{path}"
    if file_url in processed:
        return []

    api_url = f"{API_BASE}/repos/{owner}/{repo}/contents/{path}"
    data = api_get(session, api_url, params={"ref": branch})
    mark_processed(file_url, PROCESSED_FILE)
    processed.add(file_url)

    if not data or not isinstance(data, dict):
        return []
    if data.get("size", 0) > MAX_FILE_SIZE_BYTES:
        return []

    raw_content = data.get("content", "")
    if not raw_content:
        return []

    try:
        decoded = base64.b64decode(raw_content).decode("utf-8", errors="replace")
    except Exception:
        return []

    file_sha = data.get("sha", "")
    ext = path.lower().split(".")[-1]

    if ext == "ipynb":
        tuples = extract_circuits_notebook(decoded)
    else:
        tuples = extract_circuits_python(decoded)

    entries = []
    for (code, start_line, end_line) in tuples:
        ch = hashlib.md5(code.strip().encode("utf-8")).hexdigest()
        if ch in seen_hashes:
            continue
        seen_hashes.add(ch)

        if start_line is not None and end_line is not None:
            github_anchor = f"{file_url}#L{start_line}-L{end_line}"
        else:
            github_anchor = file_url

        entry = {
            "input":          "",
            "output":         code,
            "openqasm3_code": None,
            "metadata": {
                # Cluster 1 — Provenance
                "original_url":  file_url,
                "file_path":     path,
                "source":        source_tag,
                "language":      "jupyter" if ext == "ipynb" else "python",
                "circuit_hash":  ch,
                "content_hash":  None,
                "hash":          file_sha,
                "start_line":    start_line,
                "end_line":      end_line,
                "github_anchor": github_anchor,
                "repo_owner":    owner,
                "repo_name":     repo,
                "scrape_date":   SCRAPE_DATE,
                "code_lines":    len([l for l in code.splitlines() if l.strip()]),
                "output_token_count_cl100k":  None,
                
                # Cluster 2 — Instruction Generation
                "prompt_type":               None,
                "quality_flag":              None,
                "generation_model":          None,
                "generation_date":           None,
                "paraphrase_source":         None,
                "original_prompt":           None,
                "prompt_word_count":         None,
                "prompt_length_chars":       None,
                "prompt_token_count_cl100k": None,
                
                # Cluster 3 — Repo Context
                "repo_topics":   None,
                "is_org_repo":   None,
                
                # Cluster 4 — Execution / Validation
                "validation_status":          None,
                "validation_error_type":      None,
                "circuit_stats_available":    None,
                "openqasm3_export_successful": None,
                "openqasm3_export_error":     None,
                "qiskit_version":             None,
                "api_deprecated_usage":       None,
                "deprecated_api_patterns":    None,
                "hallucination_type":         None,
                
                # Cluster 5 — Structural Circuit Metrics
                # Core Circuit Metrics
                "num_qubits":          None,
                "num_clbits":          None,
                "quantum_register_count": None,
                "gate_count":          None,
                "circuit_depth":       None,
                "circuit_width":       None,
                "gate_types":          None,
                "num_gate_types":      None,
                "avg_gates_per_layer": None,
                "has_measurement":     None,
                "is_parameterized":    None,
                "multi_qubit_gate_count": None,
                "has_control_flow":       None,
                "control_flow_op_count":  None,
                "t_count":             None,
                "t_depth":             None,
                "unconnected_qubit_count": None,
                
                # Gate-set Profile Flags
                "has_clifford_only":    None,
                "has_clifford_t":       None,
                "has_rotation_gates":   None,
                "has_entangling_gates": None,
                "has_barriers":         None,
                "has_custom_gates":     None,
                "is_unitary":           None,
                "gate_set_diversity":   None,
                
                # Cluster 6 — XAI Complexity Indicators
                "circuit_expressiveness": None,
                "size_class":             None,
                "benchmark_difficulty":   None,
                
                # Cluster 7 — Entanglement Features
                "two_qubit_gate_count":  None,
                "entangling_gate_ratio": None,
                "entanglement_depth":    None,
                
                # Cluster 8 — Parameterization Features
                "num_parameters":    None,
                "parameter_density": None,
                "parameter_reuse":   None,
                
                # Cluster 9 — Measurement / Output Structure
                "measurement_count":        None,
                "measured_qubit_count":     None,
                "reset_usage":              None,
                "mid_circuit_measurement":  None,
                "classical_register_count": None,
                
                # Cluster 10 — Topology / Interaction Graph
                "interaction_graph_edges": None,
                "graph_density":           None,
                "max_qubit_degree":        None,
                "connected_components":    None,
                
                # Cluster 11 — Transpilation Metrics
                "transpiled_depth":               None,
                "transpiled_gate_count":          None,
                "transpiled_cx_count":            None,
                "transpiled_single_qubit_count":  None,
                "transpilation_overhead":         None,
                "transpilation_successful":       None,
                "transpilation_basis_gates":      None,
                "transpilation_depth_ratio":      None,
                
                # Cluster 12 — License Fields
                "repo_license":     None,
                "license_category": None,
                
                # Cluster 13 — Circuit Family
                "circuit_family":  None,
                "semantic_intent": None,
                
                # Cluster 14 — Semantic Consistency
                "semantic_similarity_to_seed": None,
                "bert_score_f1":               None,
                "bleu_score_to_seed":          None,
                "rouge_l_to_seed":             None,
                "normalized_edit_distance":    None,
            },
        }
        entries.append(entry)
    return entries


def process_repo(session, owner: str, repo: str, source_tag: str,
                seen_hashes: set, processed: set) -> int:
    """Process all .py/.ipynb files in a repo. Returns count of new circuits."""
    branch = get_default_branch(session, owner, repo)
    files  = get_repo_py_files(session, owner, repo, branch)
    count  = 0
    for path in files:
        for e in fetch_file_circuits(
            session, owner, repo, path, branch,
            source_tag, seen_hashes, processed,
        ):
            append_circuit(e, OUTPUT_FILE)
            count += 1
    return count


print("Repo/file fetching functions defined.")

## Cell 5 — Resume-State Initialization

This cell initializes the runtime state required before any acquisition strategy can be executed. It loads the GitHub token, constructs the authenticated session, and restores persisted state such as processed URLs and previously seen circuit hashes.

Its role is to transition the notebook from definition to execution. In practical terms, this cell should be rerun after any kernel restart prior to resuming a strategy cell, so that the acquisition logic continues from the persisted state rather than behaving as a fresh run.

**Principal runtime variables created in the code cell**

- `token`: authenticated GitHub access token
- `session`: shared HTTP session reused by all acquisition strategies
- `processed`: set of already-visited file URLs
- `seen_hashes`: set of already-seen circuit hashes used for deduplication
- `me`: authenticated user record returned by the GitHub API


In [ ]:
token       = load_token()
session     = make_session(token)
processed   = load_processed(PROCESSED_FILE)
seen_hashes = load_seen_hashes(OUTPUT_FILE)

print(f"Processed files  : {len(processed):,}")
print(f"Seen circuit hashes: {len(seen_hashes):,}")

# Verify token works
me = api_get(session, f"{API_BASE}/user")
if me:
    print(f"Authenticated as : {me.get('login', '?')}")
    limits = api_get(session, f"{API_BASE}/rate_limit")
    if limits:
        core   = limits["resources"]["core"]
        search = limits["resources"]["search"]
        print(f"Core API         : {core['remaining']}/{core['limit']} remaining")
        print(f"Search API       : {search['remaining']}/{search['limit']} remaining")
else:
    print("WARNING: Could not authenticate — check token.")

## Cell 6 — Strategy 1: Curated Repository Acquisition

This strategy processes the repositories listed in `github_urls.txt`. These repositories constitute the most deliberately selected portion of the GitHub source layer and generally provide the highest precision within the baseline acquisition stage.

The rationale for starting here is methodological as well as practical. Curated repositories provide a strong initial pool of circuits from known relevant sources, thereby establishing a baseline raw corpus against which broader and noisier strategies can later be interpreted.

**Principal function and counters used in the code cell**

- `extract_owner_repo()`: normalizes curated GitHub URLs into `(owner, repo)` pairs
- `repos_seen` and `repos`: de-duplicated curated repository list
- `total_s1`: cumulative number of circuits added by the curated-repository stage


In [ ]:
import time as _time

def extract_owner_repo(url: str):
    try:
        url = url.strip()
        if not url.startswith("https://github.com/") and not url.startswith("github.com/"):
            return None
        path = url.replace("https://github.com/", "").replace("github.com/", "")
        parts = [p for p in path.split("/") if p]
        if len(parts) >= 2:
            return parts[0], parts[1]
    except Exception:
        pass
    return None


print("[Strategy 1] Curated repos from github_urls.txt", flush=True)
t0 = _time.time()

with open(GITHUB_URLS_FILE, encoding="utf-8") as f:
    raw_urls = [l.strip() for l in f if l.strip()]

repos_seen = set()
repos = []
for url in raw_urls:
    pair = extract_owner_repo(url)
    if pair and pair not in repos_seen:
        repos_seen.add(pair)
        repos.append(pair)

print(f"  Unique repos: {len(repos)}")
total_s1 = 0

for i, (owner, repo) in enumerate(repos, 1):
    n = process_repo(session, owner, repo, "curated", seen_hashes, processed)
    total_s1 += n
    if n or i % 10 == 0:
        elapsed = _time.time() - t0
        print(f"  [{i}/{len(repos)}] {owner}/{repo} → +{n}  "
              f"(total={total_s1}, elapsed={elapsed:.0f}s)", flush=True)

elapsed = _time.time() - t0
print(f"\nStrategy 1 done: {total_s1:,} new circuits in {elapsed:.0f}s")

## Cell 7 — Strategy 2: Targeted GitHub Code Search

This strategy executes targeted GitHub Code Search queries designed to recover circuit-bearing files that are not necessarily reachable through the curated repository list. The query set emphasizes Qiskit-specific construction patterns and code idioms associated with circuit definition and manipulation.

This stage generally offers broader recall than the curated repository pass, but at the cost of greater noise and slower execution. It is therefore the principal high-latency component of the baseline acquisition phase and should be interpreted as a controlled recall mechanism rather than a precision-oriented one.

**Principal counters used in the code cell**

- `total_s2`: cumulative number of circuits added by the code-search stage
- `query_total`: per-query addition count used to interpret query-level yield
- `SEARCH_QUERIES`: targeted GitHub Code Search query pack defined in the global configuration


In [ ]:
import time as _time

print("[Strategy 2] GitHub Code Search", flush=True)
t0 = _time.time()
total_s2 = 0

for qi, query in enumerate(SEARCH_QUERIES, 1):
    print(f"  Query {qi}/{len(SEARCH_QUERIES)}: {query[:70]}", flush=True)
    page = 1
    query_total = 0

    while page <= 10:
        data = api_get(
            session,
            f"{API_BASE}/search/code",
            params={"q": query, "per_page": 100, "page": page},
            sleep=SEARCH_SLEEP,
        )
        if not data:
            break

        items = data.get("items", [])
        if not items:
            break

        for item in items:
            repo_info = item.get("repository", {})
            owner = repo_info.get("owner", {}).get("login", "")
            repo  = repo_info.get("name", "")
            path  = item.get("path", "")
            if not (owner and repo and path):
                continue
            branch = get_default_branch(session, owner, repo)
            for e in fetch_file_circuits(
                session, owner, repo, path, branch,
                "search", seen_hashes, processed,
            ):
                append_circuit(e, OUTPUT_FILE)
                query_total += 1
                total_s2   += 1

        if len(items) < 100:
            break
        page += 1

    elapsed = _time.time() - t0
    print(f"    → {query_total} circuits  (running total={total_s2}, elapsed={elapsed:.0f}s)",
          flush=True)

elapsed = _time.time() - t0
print(f"\nStrategy 2 done: {total_s2:,} new circuits in {elapsed:.0f}s")

## Cell 8 — Strategy 3: Organization-Level Repository Enumeration

This strategy enumerates and processes public repositories from selected organizations, principally within the Qiskit ecosystem. It is intended to recover official and community-maintained repositories that may not have been explicitly included in the curated source list.

Relative to code search, this strategy often produces cleaner repository-level context and more coherent provenance. Relative to the curated repository stage, it offers additional breadth at modestly lower precision.

**Principal function and counters used in the code cell**

- `get_org_repos()`: enumerates public repositories for a selected organization
- `total_s3`: cumulative number of circuits added by the organization-repository stage
- `ORGS`: organization list used in the baseline organization sweep


In [ ]:
import time as _time

def get_org_repos(session, org: str) -> list:
    repos = []
    page = 1
    while True:
        data = api_get(
            session,
            f"{API_BASE}/orgs/{org}/repos",
            params={"type": "public", "per_page": 100, "page": page},
        )
        if not data or not isinstance(data, list):
            break
        repos.extend(r["name"] for r in data if not r.get("archived"))
        if len(data) < 100:
            break
        page += 1
    return repos


print("[Strategy 3] Org repos", flush=True)
t0 = _time.time()
total_s3 = 0

for org in ORGS:
    print(f"  Org: {org}", flush=True)
    repo_names = get_org_repos(session, org)
    print(f"    Repos found: {len(repo_names)}", flush=True)

    for i, repo_name in enumerate(repo_names, 1):
        n = process_repo(session, org, repo_name, "org", seen_hashes, processed)
        total_s3 += n
        if n:
            elapsed = _time.time() - t0
            print(f"    [{i}/{len(repo_names)}] {org}/{repo_name} → +{n}  "
                  f"(total={total_s3}, elapsed={elapsed:.0f}s)", flush=True)

elapsed = _time.time() - t0
print(f"\nStrategy 3 done: {total_s3:,} new circuits in {elapsed:.0f}s")

## Cell 9 — Strategy 4: Topic-Based Repository Discovery

This strategy identifies repositories through GitHub topic tags related to Qiskit and quantum computing, and then processes each retrieved repository in full. Topic-based discovery extends acquisition beyond the official ecosystem and into smaller or less centrally linked repositories.

Because repository topics are user-assigned and uneven in quality, this strategy functions as a broader recall layer with correspondingly greater heterogeneity. Its main value lies in increasing source diversity prior to the more explicitly append-only expansions introduced later.

**Principal function and counters used in the code cell**

- `get_topic_repos()`: discovers repositories through GitHub topic search
- `total_s4`: cumulative number of circuits added by the topic-repository stage
- `TOPICS`: topic list used to broaden source discovery beyond curated and organization-level sources


In [ ]:
import time as _time

def get_topic_repos(session, topic: str) -> list:
    repos = []
    page = 1
    while page <= 3:
        data = api_get(
            session,
            f"{API_BASE}/search/repositories",
            params={
                "q":        f"topic:{topic} language:python",
                "sort":     "stars",
                "order":    "desc",
                "per_page": 100,
                "page":     page,
            },
            sleep=SEARCH_SLEEP,
        )
        if not data:
            break
        items = data.get("items", [])
        for item in items:
            owner = item.get("owner", {}).get("login", "")
            name  = item.get("name", "")
            if owner and name and not item.get("archived"):
                repos.append((owner, name))
        if len(items) < 100:
            break
        page += 1
    return repos


print("[Strategy 4] Topic repos", flush=True)
t0 = _time.time()
total_s4 = 0

for topic in TOPICS:
    print(f"  Topic: {topic}", flush=True)
    repo_pairs = get_topic_repos(session, topic)
    print(f"    Repos found: {len(repo_pairs)}", flush=True)

    for i, (owner, repo) in enumerate(repo_pairs, 1):
        n = process_repo(session, owner, repo, "topic", seen_hashes, processed)
        total_s4 += n
        if n:
            elapsed = _time.time() - t0
            print(f"    [{i}/{len(repo_pairs)}] {owner}/{repo} → +{n}  "
                  f"(total={total_s4}, elapsed={elapsed:.0f}s)", flush=True)

elapsed = _time.time() - t0
print(f"\nStrategy 4 done: {total_s4:,} new circuits in {elapsed:.0f}s")

## Cell 10 — Baseline Acquisition Summary

This summary cell consolidates the outputs of the four baseline acquisition strategies. It provides a compact checkpoint on the size and composition of the initial raw pool before the notebook transitions to the append-only aggressive recall stages.

From a reproducibility perspective, this cell marks the end of the baseline GitHub acquisition phase. Subsequent expansions should be interpreted as additions layered over this baseline rather than replacements for it.

**Principal summary variables used in the code cell**

- `_s1`, `_s2`, `_s3`, `_s4`: strategy-level totals recovered from the global namespace
- `out_count`: final number of entries currently present in the baseline output file


In [ ]:
_g = globals()
_s1 = _g.get("total_s1", 0)
_s2 = _g.get("total_s2", 0)
_s3 = _g.get("total_s3", 0)
_s4 = _g.get("total_s4", 0)

out_count = 0
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, encoding="utf-8") as f:
        out_count = sum(1 for _ in f)

print("=" * 55)
print("SCRAPING COMPLETE")
print("=" * 55)
print(f"  Strategy 1 (curated)   : {_s1:,}")
print(f"  Strategy 2 (search)    : {_s2:,}")
print(f"  Strategy 3 (orgs)      : {_s3:,}")
print(f"  Strategy 4 (topics)    : {_s4:,}")
print(f"  New circuits this run  : {_s1+_s2+_s3+_s4:,}")
print(f"  Total in output file   : {out_count:,}")
print(f"  Processed file URLs    : {len(processed):,}")
print()
print("Next step: run enrich_raw_circuits.py on circuits_unified.jsonl")


# Phase 2 Aggressive Rescrape

Phase 2 is an append-only aggressive rescrape designed to extend recall without invalidating the baseline outputs. The central principle of this phase is that broader acquisition should remain auditable: Phase 2 writes to its own files, preserves the baseline artifacts, and explicitly annotates retrieval provenance so that downstream analysis can distinguish where each circuit entered the corpus.

This phase therefore introduces and propagates three provenance fields:

- `retrieval_mode`
- `retrieval_strategy`
- `retrieval_run_id`

The resulting design permits later comparisons between baseline acquisition, aggressive expansion, and the final Phase 3 high-yield recovery pass.


## Cell 11 — Phase 2 Configuration and Isolated Resume State

This cell defines the output paths, manifest location, retrieval labels, and isolated processed-state files used by Phase 2. By assigning Phase 2 its own output and resume artifacts, the notebook ensures that the aggressive rescrape remains analytically separable from the baseline.

This separation is methodologically useful because it allows the contribution of Phase 2 to be measured explicitly during later merges and benchmark-tier analyses.

**Principal variables introduced in the code cell**

- `PHASE2_LABEL`, `RETRIEVAL_MODE_V2`, `RUN_ID_V2`: identifiers used to track Phase 2 provenance
- `OUTPUT_FILE_V2`, `PROCESSED_FILE_V2`, `MERGED_FILE_V2`, `MANIFEST_FILE_V2`: Phase 2 artifacts
- `PHASE2_PROMOTED_REPOS`, `PHASE2_ORGS`, `PHASE2_SEARCH_QUERIES`: the three main Phase 2 source lists

**Principal helper functions defined in the code cell**

- `load_jsonl_any()`: generic JSONL reader used throughout the append-only phases
- `repo_rel_display()`: publication-safe relative-path display helper
- `append_jsonl_to()` and `mark_processed_to()`: file helpers scoped to the append-only stages
- `write_manifest_v2()`: writes the Phase 2 manifest
- `metadata_template_v2()`: constructs Phase 2 retrieval-aware metadata payloads


In [ ]:
# Cell 11 — Phase 2 config, manifest, and isolated state

from collections import Counter

PHASE2_LABEL = "aggressive_v1"
RETRIEVAL_MODE_V2 = "aggressive"
RUN_ID_V2 = f"{PHASE2_LABEL}_{SCRAPE_DATE}"

OUTPUT_FILE_V2 = BASE / "circuits_unified_aggressive.jsonl"
PROCESSED_FILE_V2 = BASE / "circuits_unified_aggressive_processed.txt"
MERGED_FILE_V2 = BASE / "circuits_unified_plus_aggressive.jsonl"
MANIFEST_FILE_V2 = BASE / "circuits_unified_aggressive_manifest.json"

MAX_FILE_SIZE_BYTES_V2 = 1_000_000
INCLUDE_DOC_PATHS_V2 = True

SKIP_PATH_SEGMENTS_V2 = (
    "__pycache__", ".egg-info", "node_modules", "build/", "dist/", ".tox/",
)
PREFERRED_PATH_HINTS_V2 = (
    "examples/", "example/", "notebooks/", "tutorial", "demo", "docs/",
)

AGGRESSIVE_CIRCUIT_HINTS = (
    "TwoLocal(", "RealAmplitudes(", "EfficientSU2(", "QFT(",
    "ZZFeatureMap(", "PauliFeatureMap(", "NLocal(", "BlueprintCircuit(",
    "QuantumCircuit.from_qasm_str(", ".compose(", ".append(", ".decompose(",
    "measure_all(", "Sampler(", "Estimator(",
)

CIRCUIT_CONTINUATION_TOKENS_V2 = (
    ".h(", ".cx(", ".ccx(", ".measure", ".barrier", ".ry(", ".rz(", ".rx(",
    ".x(", ".y(", ".z(", ".s(", ".t(", ".p(", ".u(", ".swap(", ".cz(",
    ".ch(", ".cp(", ".append(", ".compose(", ".decompose(", ".draw(",
    ".transpile(", "qc.", "circ.", "circuit.", "ansatz", "feature_map",
    "wavefunction", "var_form", "QuantumRegister(", "ClassicalRegister(",
    "ParameterVector(", "Parameter(",
)

PHASE2_PROMOTED_REPOS = [
    ("PennyLaneAI", "pennylane-qiskit"),
    ("Quantinuum", "pytket-qiskit"),
]

PHASE2_ORGS = [
    "PennyLaneAI",
    "Quantinuum",
]

PHASE2_SEARCH_QUERIES = list(dict.fromkeys([
    "from qiskit.circuit.library import TwoLocal language:python",
    "from qiskit.circuit.library import RealAmplitudes language:python",
    "from qiskit.circuit.library import EfficientSU2 language:python",
    "from qiskit.circuit.library import QFT language:python",
    "from qiskit.circuit.library import ZZFeatureMap language:python",
    "from qiskit.circuit.library import PauliFeatureMap language:python",
    "qc.compose( language:python",
    "qc.append( language:python",
    "measure_all() qiskit language:jupyter-notebook",
]))

SCHEMA_NULL_FIELDS = [
    "content_hash",
    "prompt_type", "quality_flag", "generation_model", "generation_date",
    "paraphrase_source", "original_prompt", "prompt_word_count",
    "prompt_length_chars", "prompt_token_count_cl100k",
    "repo_topics", "is_org_repo",
    "validation_status", "validation_error_type", "circuit_stats_available",
    "openqasm3_export_successful", "openqasm3_export_error", "qiskit_version",
    "api_deprecated_usage", "deprecated_api_patterns", "hallucination_type",
    "num_qubits", "num_clbits", "quantum_register_count", "gate_count",
    "circuit_depth", "circuit_width", "gate_types", "num_gate_types",
    "avg_gates_per_layer", "has_measurement", "is_parameterized",
    "multi_qubit_gate_count", "has_control_flow", "control_flow_op_count",
    "t_count", "t_depth", "unconnected_qubit_count",
    "has_clifford_only", "has_clifford_t", "has_rotation_gates",
    "has_entangling_gates", "has_barriers", "has_custom_gates", "is_unitary",
    "gate_set_diversity", "circuit_expressiveness", "size_class",
    "benchmark_difficulty", "two_qubit_gate_count", "entangling_gate_ratio",
    "entanglement_depth", "num_parameters", "parameter_density",
    "parameter_reuse", "measurement_count", "measured_qubit_count",
    "reset_usage", "mid_circuit_measurement", "classical_register_count",
    "interaction_graph_edges", "graph_density", "max_qubit_degree",
    "connected_components", "transpiled_depth", "transpiled_gate_count",
    "transpiled_cx_count", "transpiled_single_qubit_count",
    "transpilation_overhead", "transpilation_successful",
    "transpilation_basis_gates", "transpilation_depth_ratio",
    "repo_license", "license_category", "circuit_family", "semantic_intent",
    "semantic_similarity_to_seed", "bert_score_f1", "bleu_score_to_seed",
    "rouge_l_to_seed", "normalized_edit_distance",
    "output_token_count_cl100k",
]

def load_jsonl_any(path):
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def repo_rel_display(path):
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT))
    except Exception:
        try:
            return str(Path(path))
        except Exception:
            return str(path)

def append_jsonl_to(entry, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def mark_processed_to(url, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(url + "\n")

def write_manifest_v2(extra=None):
    def _repo_rel(path):
        try:
            return str(Path(path).resolve().relative_to(REPO_ROOT))
        except Exception:
            try:
                return str(Path(path))
            except Exception:
                return str(path)

    manifest = {
        "phase": 2,
        "label": PHASE2_LABEL,
        "retrieval_mode": RETRIEVAL_MODE_V2,
        "retrieval_run_id": RUN_ID_V2,
        "scrape_date": SCRAPE_DATE,
        "baseline_output_file": _repo_rel(OUTPUT_FILE),
        "aggressive_output_file": _repo_rel(OUTPUT_FILE_V2),
        "merged_output_file": _repo_rel(MERGED_FILE_V2),
        "processed_file_v2": _repo_rel(PROCESSED_FILE_V2),
        "max_file_size_bytes_v2": MAX_FILE_SIZE_BYTES_V2,
        "include_doc_paths_v2": INCLUDE_DOC_PATHS_V2,
        "promoted_repos_v2": PHASE2_PROMOTED_REPOS,
        "orgs_v2": PHASE2_ORGS,
        "search_queries_v2": PHASE2_SEARCH_QUERIES,
    }
    if extra:
        manifest.update(extra)
    with open(MANIFEST_FILE_V2, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

def metadata_template_v2(
    owner, repo, path, file_url, file_sha, ext,
    start_line, end_line, code, source_tag, retrieval_strategy
):
    meta = {k: None for k in SCHEMA_NULL_FIELDS}
    github_anchor = file_url if start_line is None else f"{file_url}#L{start_line}-L{end_line}"
    meta.update({
        "original_url": file_url,
        "file_path": path,
        "source": source_tag,
        "language": "jupyter" if ext == "ipynb" else "python",
        "circuit_hash": hashlib.md5(code.strip().encode("utf-8")).hexdigest(),
        "hash": file_sha,
        "start_line": start_line,
        "end_line": end_line,
        "github_anchor": github_anchor,
        "repo_owner": owner,
        "repo_name": repo,
        "scrape_date": SCRAPE_DATE,
        "code_lines": len([l for l in code.splitlines() if l.strip()]),
        "retrieval_mode": RETRIEVAL_MODE_V2,
        "retrieval_strategy": retrieval_strategy,
        "retrieval_run_id": RUN_ID_V2,
    })
    return meta

processed_v2 = load_processed(PROCESSED_FILE_V2)
seen_hashes_v2 = load_seen_hashes(OUTPUT_FILE) | load_seen_hashes(OUTPUT_FILE_V2)

write_manifest_v2()

print(f"Phase 2 label       : {PHASE2_LABEL}")
print(f"Phase 2 output      : {repo_rel_display(OUTPUT_FILE_V2)}")
print(f"Phase 2 processed   : {repo_rel_display(PROCESSED_FILE_V2)}")
print(f"Phase 2 merged file : {repo_rel_display(MERGED_FILE_V2)}")
print(f"Combined seen hashes: {len(seen_hashes_v2):,}")


## Cell 12 — Phase 2 Helper Wrappers

This cell defines the extraction and fetch wrappers specific to the aggressive rescrape. These wrappers extend the earlier baseline helpers while stamping Phase 2-specific retrieval provenance into each extracted entry.

The cell therefore functions as the local operational scaffold for all subsequent Phase 2 strategy cells.

**Principal functions defined in the code cell**

- `_extract_function_blocks_aggressive()` and `_extract_module_level_blocks_aggressive()`: more permissive extraction logic for aggressive recall
- `extract_circuits_python_aggressive()` and `extract_circuits_notebook_aggressive()`: aggressive source extractors
- `get_repo_code_files_v2()`: Phase 2 repository file enumerator
- `get_org_repos_v2()`: Phase 2 organization-repository enumerator
- `fetch_file_circuits_v2()`: Phase 2 file fetch-and-extract helper
- `process_repo_v2()`: Phase 2 repository-level processing wrapper


In [ ]:
# Cell 12 — Aggressive extraction + Phase 2 fetch wrappers
def _extract_function_blocks_aggressive(lines):
    blocks = []
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        if re.match(r"^def\s+\w+\s*\(", stripped):
            start_idx = i
            indent = len(line) - len(line.lstrip())
            body_lines = [line]
            j = i + 1
            while j < len(lines):
                next_line = lines[j]
                if next_line.strip() == "":
                    body_lines.append(next_line)
                    j += 1
                    continue
                next_indent = len(next_line) - len(next_line.lstrip())
                if next_indent > indent:
                    body_lines.append(next_line)
                    j += 1
                else:
                    break
            body = "\n".join(body_lines).rstrip()
            if (
                ("QuantumCircuit(" in body or any(h in body for h in AGGRESSIVE_CIRCUIT_HINTS))
                and len(body.split()) >= MIN_CIRCUIT_TOKENS
            ):
                blocks.append((body, start_idx + 1, j))
            i = j
        else:
            i += 1
    return blocks

def _extract_module_level_blocks_aggressive(lines):
    blocks = []
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        if (
            not stripped
            or stripped.startswith("#")
            or stripped.startswith("import ")
            or stripped.startswith("from ")
            or re.match(r"^(class|def)\s", stripped)
        ):
            i += 1
            continue

        starts_block = (
            not line.startswith((" ", "\t"))
            and (
                "QuantumCircuit(" in stripped
                or any(h in stripped for h in AGGRESSIVE_CIRCUIT_HINTS)
            )
        )

        if starts_block:
            start_idx = i
            block_lines = [line]
            j = i + 1
            while j < len(lines):
                nl = lines[j]
                ns = nl.strip()

                if not ns or ns.startswith("#"):
                    block_lines.append(nl)
                    j += 1
                    continue

                if nl.startswith((" ", "\t")):
                    block_lines.append(nl)
                    j += 1
                    continue

                if (
                    "QuantumCircuit(" in ns
                    or any(h in ns for h in AGGRESSIVE_CIRCUIT_HINTS)
                    or any(tok in ns for tok in CIRCUIT_CONTINUATION_TOKENS_V2)
                ):
                    block_lines.append(nl)
                    j += 1
                    continue

                break

            block = "\n".join(block_lines).rstrip()
            if len(block.split()) >= MIN_CIRCUIT_TOKENS:
                blocks.append((block, start_idx + 1, j))
            i = j
        else:
            i += 1

    return blocks

def extract_circuits_python_aggressive(code):
    if not any(h in code for h in ("QuantumCircuit",) + AGGRESSIVE_CIRCUIT_HINTS):
        return []

    lines = code.splitlines()
    results = []
    seen_blocks = set()

    for tup in _extract_function_blocks_aggressive(lines):
        if tup[0] not in seen_blocks:
            results.append(tup)
            seen_blocks.add(tup[0])

    for tup in _extract_module_level_blocks_aggressive(lines):
        if tup[0] not in seen_blocks:
            results.append(tup)
            seen_blocks.add(tup[0])

    return results

def extract_circuits_notebook_aggressive(raw_json):
    try:
        nb = json.loads(raw_json)
    except Exception:
        return []

    results = []
    seen_blocks = set()

    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)

        for blk, _sl, _el in extract_circuits_python_aggressive(src):
            if blk not in seen_blocks:
                results.append((blk, None, None))
                seen_blocks.add(blk)

    return results

def get_repo_code_files_v2(session, owner, repo, branch):
    tree_url = f"{API_BASE}/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    data = api_get(session, tree_url)
    if not data:
        return []

    files = []
    for item in data.get("tree", []):
        path = item.get("path", "")
        lower = path.lower()
        if item.get("type") != "blob":
            continue
        if not lower.endswith((".py", ".ipynb")):
            continue
        if item.get("size", 0) > MAX_FILE_SIZE_BYTES_V2:
            continue
        if any(seg.lower() in lower for seg in SKIP_PATH_SEGMENTS_V2):
            continue
        if (not INCLUDE_DOC_PATHS_V2) and "docs/" in lower:
            continue
        files.append(path)

    files.sort(
        key=lambda p: (
            0 if any(h in p.lower() for h in PREFERRED_PATH_HINTS_V2) else 1,
            p.lower(),
        )
    )
    return files

def get_org_repos_v2(session, org):
    repos = []
    page = 1
    while True:
        data = api_get(
            session,
            f"{API_BASE}/orgs/{org}/repos",
            params={"type": "public", "per_page": 100, "page": page},
        )
        if not data or not isinstance(data, list):
            break
        repos.extend(r["name"] for r in data if not r.get("archived"))
        if len(data) < 100:
            break
        page += 1
    return repos

def fetch_file_circuits_v2(
    session, owner, repo, path, branch,
    source_tag, retrieval_strategy,
    seen_hashes, processed, processed_file
):
    file_url = f"https://github.com/{owner}/{repo}/blob/{branch}/{path}"
    if file_url in processed:
        return []

    api_url = f"{API_BASE}/repos/{owner}/{repo}/contents/{path}"
    data = api_get(session, api_url, params={"ref": branch})
    mark_processed_to(file_url, processed_file)
    processed.add(file_url)

    if not data or not isinstance(data, dict):
        return []
    if data.get("size", 0) > MAX_FILE_SIZE_BYTES_V2:
        return []

    raw_content = data.get("content", "")
    if not raw_content:
        return []

    try:
        decoded = base64.b64decode(raw_content).decode("utf-8", errors="replace")
    except Exception:
        return []

    file_sha = data.get("sha", "")
    ext = path.lower().split(".")[-1]

    if ext == "ipynb":
        tuples = extract_circuits_notebook_aggressive(decoded)
    else:
        tuples = extract_circuits_python_aggressive(decoded)

    entries = []
    for code, start_line, end_line in tuples:
        circuit_hash = hashlib.md5(code.strip().encode("utf-8")).hexdigest()
        if circuit_hash in seen_hashes:
            continue
        seen_hashes.add(circuit_hash)

        entry = {
            "input": "",
            "output": code,
            "openqasm3_code": None,
            "metadata": metadata_template_v2(
                owner=owner,
                repo=repo,
                path=path,
                file_url=file_url,
                file_sha=file_sha,
                ext=ext,
                start_line=start_line,
                end_line=end_line,
                code=code,
                source_tag=source_tag,
                retrieval_strategy=retrieval_strategy,
            ),
        }
        entries.append(entry)

    return entries

def process_repo_v2(
    session, owner, repo,
    source_tag, retrieval_strategy,
    seen_hashes, processed, processed_file, output_file
):
    branch = get_default_branch(session, owner, repo)
    files = get_repo_code_files_v2(session, owner, repo, branch)
    count = 0

    for path in files:
        for entry in fetch_file_circuits_v2(
            session, owner, repo, path, branch,
            source_tag, retrieval_strategy,
            seen_hashes, processed, processed_file,
        ):
            append_jsonl_to(entry, output_file)
            count += 1

    return count

print("Phase 2 extraction and fetch wrappers ready.")

## Cell 13 — Empirical Promotion Suggestions

This diagnostic cell derives optional empirical promotion suggestions from the baseline outputs. Its purpose is to identify repositories that appear likely to contribute substantial additional yield but that should be reviewed explicitly before being promoted into the aggressive rescrape.

The cell is therefore advisory rather than transformative: it supports manual judgment without itself altering the dataset.

**Principal function and counters used in the code cell**

- `extract_owner_repo_pair()`: normalizes repository URLs to `(owner, repo)` pairs
- `owner_counts_v2` and `repo_counts_v2`: empirical frequency summaries used to suggest promoted repositories


In [ ]:
# Cell 13 — Optional: empirical promotion suggestions from baseline output
def extract_owner_repo_pair(url):
    url = url.strip()
    if not url.startswith("https://github.com/"):
        return None
    parts = [p for p in url.replace("https://github.com/", "").split("/") if p]
    if len(parts) >= 2:
        return parts[0], parts[1]
    return None

curated_repo_pairs = set()
if GITHUB_URLS_FILE.exists():
    with open(GITHUB_URLS_FILE, encoding="utf-8") as f:
        for line in f:
            pair = extract_owner_repo_pair(line)
            if pair:
                curated_repo_pairs.add(pair)

owner_counts_v2 = Counter()
repo_counts_v2 = Counter()

for entry in load_jsonl_any(OUTPUT_FILE):
    meta = entry.get("metadata", {})
    owner = meta.get("repo_owner")
    repo = meta.get("repo_name")
    if not owner or not repo:
        continue
    if (owner, repo) in curated_repo_pairs:
        continue
    owner_counts_v2[owner] += 1
    repo_counts_v2[(owner, repo)] += 1

print("Top non-curated repo candidates:")
for (owner, repo), n in repo_counts_v2.most_common(15):
    print(f"  {owner}/{repo} -> {n}")

print("\nTop owner candidates:")
for owner, n in owner_counts_v2.most_common(15):
    print(f"  {owner} -> {n}")

## Cell 14 — Strategy A: Promoted Repositories

This strategy executes the aggressive rescrape over a manually selected set of promoted repositories. These repositories are expected to provide a meaningful incremental contribution beyond the baseline, while remaining more interpretable than an unconstrained broad sweep.

In methodological terms, this strategy represents a targeted precision-recall trade-off: broader than the baseline, but still grounded in deliberate repository selection.

**Principal counters used in the code cell**

- `total_promoted_v2`: cumulative number of circuits added by promoted repositories
- `promoted_counts_v2`: per-repository yield summary for the promoted-repository stage


In [ ]:
# Cell 14 — Strategy A: promoted repos
total_promoted_v2 = 0
promoted_counts_v2 = {}

for i, (owner, repo) in enumerate(PHASE2_PROMOTED_REPOS, 1):
    n = process_repo_v2(
        session, owner, repo,
        source_tag="promoted_repo_v2",
        retrieval_strategy="promoted_repo",
        seen_hashes=seen_hashes_v2,
        processed=processed_v2,
        processed_file=PROCESSED_FILE_V2,
        output_file=OUTPUT_FILE_V2,
    )
    promoted_counts_v2[f"{owner}/{repo}"] = n
    total_promoted_v2 += n
    print(f"[{i}/{len(PHASE2_PROMOTED_REPOS)}] {owner}/{repo} -> +{n}")

print(f"\nPromoted repo total: {total_promoted_v2:,}")


## Cell 15 — Strategy B: Selective Organization Expansion

This strategy broadens Phase 2 through selective organization-level expansion. Rather than scraping all plausible owners indiscriminately, the cell restricts the expansion to a limited set of organizations judged likely to contain relevant circuit-bearing repositories.

This design preserves a measure of reviewability while still extending recall beyond the baseline organization coverage.

**Principal counters used in the code cell**

- `total_org_v2`: cumulative number of circuits added by selective organization expansion
- `org_repo_counts_v2`: repository-count summary per organization


In [ ]:
# Cell 15 — Strategy B: selective org expansion
total_org_v2 = 0
org_repo_counts_v2 = {}

for org in PHASE2_ORGS:
    repo_names = get_org_repos_v2(session, org)
    org_repo_counts_v2[org] = len(repo_names)
    print(f"{org}: {len(repo_names)} public repos")

    for i, repo_name in enumerate(repo_names, 1):
        n = process_repo_v2(
            session, org, repo_name,
            source_tag="org_v2",
            retrieval_strategy="org",
            seen_hashes=seen_hashes_v2,
            processed=processed_v2,
            processed_file=PROCESSED_FILE_V2,
            output_file=OUTPUT_FILE_V2,
        )
        total_org_v2 += n
        if n:
            print(f"  [{i}/{len(repo_names)}] {org}/{repo_name} -> +{n}")

print(f"\nSelective org total: {total_org_v2:,}")

## Cell 16 — Strategy C: Expanded Search Queries

This strategy executes the broader Phase 2 search-query pack. Its purpose is to recover circuit-bearing files that are unlikely to be found through curated sources or controlled organization-level expansion alone.

The resulting output contributes substantially to the broad raw pool later used for enrichment, audit, and preliminary broad-to-core splitting. Because the strategy is search-driven, it should be understood primarily as a recall mechanism.

**Principal counters used in the code cell**

- `total_search_v2`: cumulative number of circuits added by the expanded search stage
- `search_query_counts_v2`: per-query yield record used later when interpreting aggressive search behavior


In [ ]:
# Cell 16 — Strategy C: expanded search
total_search_v2 = 0
search_query_counts_v2 = {}

for qi, query in enumerate(PHASE2_SEARCH_QUERIES, 1):
    print(f"[{qi}/{len(PHASE2_SEARCH_QUERIES)}] {query}")
    page = 1
    query_total = 0

    while page <= 10:
        data = api_get(
            session,
            f"{API_BASE}/search/code",
            params={"q": query, "per_page": 100, "page": page},
            sleep=SEARCH_SLEEP,
        )
        if not data:
            break

        items = data.get("items", [])
        if not items:
            break

        for item in items:
            repo_info = item.get("repository", {})
            owner = repo_info.get("owner", {}).get("login", "")
            repo = repo_info.get("name", "")
            path = item.get("path", "")
            if not (owner and repo and path):
                continue

            branch = get_default_branch(session, owner, repo)
            for entry in fetch_file_circuits_v2(
                session, owner, repo, path, branch,
                source_tag="search_v2",
                retrieval_strategy="expanded_search",
                seen_hashes=seen_hashes_v2,
                processed=processed_v2,
                processed_file=PROCESSED_FILE_V2,
            ):
                append_jsonl_to(entry, OUTPUT_FILE_V2)
                query_total += 1
                total_search_v2 += 1

        if len(items) < 100:
            break
        page += 1

    search_query_counts_v2[query] = query_total
    print(f"  -> {query_total} circuits (running total={total_search_v2:,})")

print(f"\nExpanded search total: {total_search_v2:,}")


## Cell 17 — Frozen Empirical Promotions

This cell records the final manually frozen empirical promotion set used by the aggressive rescrape. It converts the optional diagnostics from Cell 13 into a deterministic acquisition decision.

This explicit freezing step is important for reproducibility because it separates exploratory suggestion from finalized acquisition policy.

**Principal variables used in the code cell**

- `EMPIRICAL_PROMOTED_REPOS`: final manually frozen empirical promotion list
- `total_empirical_v2`: cumulative yield from the empirical promotion pass
- `empirical_counts_v2`: per-repository yield summary for the frozen empirical set


In [ ]:
# Cell 17 — Manually frozen empirical promotions selected from Cell 13 output
EMPIRICAL_PROMOTED_REPOS = [
    ("runtsang", "Q-Bridge"),
    ("backordinary", "QDP-FSL"),
    ("wjy99-c", "QDiff"),
    ("Ali-hey-0", "Qiskit"),
    ("Ahmik-Virani", "Differentiating-Quantum-Bug-From-Noise-Statistical-Approach"),
    ("dereklin1205", "COMM_LAB_Final"),
    ("sethuquantum", "LearnQuantum"),
    ("HAMEEMM", "qiskit"),
    ("Arka221B", "Qiskit_terra"),
    ("Qiskit", "documentation"),
    ("AayushSarkar", "Qiskit-Experiment-Hub"),
    ("lockephi", "Allentown-L104-Node"),
    ("JoseCarlos458", "qiskit"),
    ("Simula-COMPLEX", "MutTG-paper"),
    ("AIComputing101", "quantum-computing-101"),
]

total_empirical_v2 = 0
empirical_counts_v2 = {}

for i, (owner, repo) in enumerate(EMPIRICAL_PROMOTED_REPOS, 1):
    n = process_repo_v2(
        session, owner, repo,
        source_tag="empirical_promoted_repo_v2",
        retrieval_strategy="empirical_promoted_repo",
        seen_hashes=seen_hashes_v2,
        processed=processed_v2,
        processed_file=PROCESSED_FILE_V2,
        output_file=OUTPUT_FILE_V2,
    )
    empirical_counts_v2[f"{owner}/{repo}"] = n
    total_empirical_v2 += n
    print(f"[{i}/{len(EMPIRICAL_PROMOTED_REPOS)}] {owner}/{repo} -> +{n}")

print(f"\nEmpirical promoted repo total: {total_empirical_v2:,}")


# Cell 18 — Merge Baseline and Aggressive Outputs

This merge stage combines the baseline raw pool with the Phase 2 aggressive outputs and backfills retrieval metadata so that all entries carry phase-aware provenance. The merged artifact becomes the principal broad raw pool for Phase 2 inspection and quality analysis.

The merge is analytically significant because it preserves acquisition-phase traceability while allowing downstream enrichment to operate on a single consolidated raw pool.

**Principal helper functions defined in the code cell**

- `normalize_retrieval_metadata()`: ensures every merged entry carries explicit retrieval provenance
- `merge_unique_by_circuit_hash()`: merges baseline and Phase 2 outputs while deduplicating by circuit hash

**Principal merged-state variables created in the code cell**

- `MERGED_FILE_V2_BROAD`: broad merged raw-pool file after baseline + Phase 2
- `broad_total_v2`: total unique entries in the merged broad pool
- `broad_mode_counts_v2` and `broad_strategy_counts_v2`: retrieval-provenance summaries of the merged pool


In [ ]:
# Cell 18 — Broad merge + backfill retrieval metadata
_g = globals()
total_empirical_v2 = _g.get("total_empirical_v2", 0)

def repo_rel_display(path):
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT))
    except Exception:
        try:
            return str(Path(path))
        except Exception:
            return str(path)

MERGED_FILE_V2_BROAD = BASE / "circuits_unified_plus_aggressive_broad.jsonl"

def normalize_retrieval_metadata(entry, default_mode, default_run_id):
    entry = dict(entry)
    entry["metadata"] = dict(entry.get("metadata", {}))
    meta = entry["metadata"]

    if meta.get("retrieval_mode") in (None, ""):
        meta["retrieval_mode"] = default_mode
    if meta.get("retrieval_strategy") in (None, ""):
        meta["retrieval_strategy"] = meta.get("source", "unknown")
    if meta.get("retrieval_run_id") in (None, ""):
        meta["retrieval_run_id"] = default_run_id

    return entry

def merge_unique_by_circuit_hash(sources, output_path):
    seen = set()
    total = 0

    with open(output_path, "w", encoding="utf-8") as out:
        for path, default_mode, default_run_id in sources:
            if not path.exists():
                continue

            with open(path, encoding="utf-8") as f:
                for line in f:
                    if not line.strip():
                        continue

                    entry = normalize_retrieval_metadata(
                        json.loads(line),
                        default_mode=default_mode,
                        default_run_id=default_run_id,
                    )
                    ch = entry.get("metadata", {}).get("circuit_hash", "")
                    if ch and ch in seen:
                        continue
                    if ch:
                        seen.add(ch)

                    out.write(json.dumps(entry, ensure_ascii=False) + "\n")
                    total += 1

    return total

broad_total_v2 = merge_unique_by_circuit_hash(
    [
        (OUTPUT_FILE, "baseline", "baseline_legacy"),
        (OUTPUT_FILE_V2, "aggressive", RUN_ID_V2),
    ],
    MERGED_FILE_V2_BROAD,
)

baseline_count_v2 = len(load_jsonl_any(OUTPUT_FILE))
aggressive_count_v2 = len(load_jsonl_any(OUTPUT_FILE_V2))
broad_entries_v2 = load_jsonl_any(MERGED_FILE_V2_BROAD)

broad_mode_counts_v2 = Counter(
    e.get("metadata", {}).get("retrieval_mode", "<missing>")
    for e in broad_entries_v2
)
broad_strategy_counts_v2 = Counter(
    e.get("metadata", {}).get("retrieval_strategy", "<missing>")
    for e in broad_entries_v2
)

print("=" * 55)
print("PHASE 2 BROAD MERGE COMPLETE")
print("=" * 55)
print(f"Baseline raw pool      : {baseline_count_v2:,}")
print(f"Aggressive additions   : {aggressive_count_v2:,}")
print(f"Broad merged circuits  : {broad_total_v2:,}")
print(f"Empirical repos so far : {total_empirical_v2:,}")

print("\nMerged retrieval_mode counts:")
for k, v in broad_mode_counts_v2.items():
    print(f"  {k}: {v:,}")

print("\nMerged retrieval_strategy counts:")
for k, v in broad_strategy_counts_v2.most_common():
    print(f"  {k}: {v:,}")

write_manifest_v2({
    "baseline_count": baseline_count_v2,
    "aggressive_count": aggressive_count_v2,
    "broad_merged_count": broad_total_v2,
    "empirical_repo_total_so_far": total_empirical_v2,
    "broad_retrieval_mode_counts": dict(broad_mode_counts_v2),
    "broad_retrieval_strategy_counts": dict(broad_strategy_counts_v2),
})

print(f"\nBroad merged file written to: {repo_rel_display(MERGED_FILE_V2_BROAD)}")
print(f"Manifest written to        : {repo_rel_display(MANIFEST_FILE_V2)}")


### Phase 2 Broad Enrichment Run

This cell runs `enrich_raw_circuits.py` over the merged Phase 2 broad raw pool. The script adds execution-aware and extraction-quality metadata without yet committing to benchmark-tier inclusion.

This stage transforms the broad merged raw pool from a purely acquisition-oriented artifact into a quality-inspectable enriched corpus suitable for later auditing and splitting.

**Principal variables used in the code cell**

- `ROOT`: repository root resolved at runtime
- `broad_raw` and `broad_enriched`: input and output files for Phase 2 broad enrichment
- `script`: path to `enrich_raw_circuits.py`


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
broad_raw = ROOT / "PQID/data/processed/circuits_unified_plus_aggressive_broad.jsonl"
broad_enriched = ROOT / "PQID/data/processed/circuits_unified_plus_aggressive_broad_enriched.jsonl"
script = ROOT / "PQID/scripts/enrich_raw_circuits.py"

print("Phase 2 broad raw present:", broad_raw.exists())
print("Phase 2 enrichment script present:", script.exists())

result = subprocess.run(
    [
        sys.executable,
        str(script),
        "--input-file",
        str(broad_raw),
        "--output-file",
        str(broad_enriched),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("Phase 2 broad enrichment completed with return code:", result.returncode)

### Phase 2 Broad Enrichment Output Check

This short cell confirms that the Phase 2 enriched broad file was written successfully before downstream audit and tiering steps are executed.

**Principal variable used in the code cell**

- `broad_enriched`: enriched Phase 2 broad-pool file whose presence is being verified


In [ ]:
print("Phase 2 broad enriched file present:", broad_enriched.exists())


### Phase 2 Broad Extraction Audit

This cell runs `report_extraction_quality.py` over the Phase 2 enriched broad pool. The resulting report provides a descriptive audit of validation outcomes, extraction confidence, cleanup candidates, and repository-level concentration of problematic entries.

The audit is intentionally read-only. Its role is diagnostic rather than transformative.

**Principal variables used in the code cell**

- `report_script`: path to `report_extraction_quality.py`
- `report_file`: markdown report written by the extraction audit
- `samples_file`: JSONL sample file written by the extraction audit


In [ ]:
report_script = ROOT / "PQID/scripts/report_extraction_quality.py"
report_file = ROOT / "PQID/data/processed/extraction_quality_report_broad.md"
samples_file = ROOT / "PQID/data/processed/extraction_quality_samples_broad.jsonl"

result = subprocess.run(
    [
        sys.executable,
        str(report_script),
        "--input-file",
        str(broad_enriched),
        "--report-file",
        str(report_file),
        "--samples-file",
        str(samples_file),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("broad extraction audit completed with return code:", result.returncode)

## Cell 19 — Broad-to-Core Split

This cell performs the initial split of the enriched broad Phase 2 pool into a strict benchmark core and a broader remainder. It applies explicit quality conditions rather than informal quality labels, thereby establishing the first benchmark-oriented partition in the notebook.

Although later Phase 3 post-processing supersedes these interim counts, this cell remains useful as a record of the earlier broad-to-core methodology.

**Principal helper functions and variables used in the code cell**

- `write_jsonl_any()`: utility for writing filtered JSONL entries
- `core_rejection_reason_v2()`: assigns interpretable rejection reasons during broad-to-core splitting
- `BROAD_ENRICHED_FILE_V2`, `CORE_ENRICHED_FILE_V2`, `TIER2_ENRICHED_FILE_V2`: Phase 2 split artifacts
- `core_rejection_counts_v2`, `core_strategy_counts_v2`, `tier2_strategy_counts_v2`: summary diagnostics for the split


In [ ]:
# Cell 19 — Split enriched broad pool into strict core and Tier 2
BROAD_ENRICHED_FILE_V2 = BASE / "circuits_unified_plus_aggressive_broad_enriched.jsonl"
CORE_ENRICHED_FILE_V2 = BASE / "circuits_unified_plus_aggressive_core_enriched.jsonl"
TIER2_ENRICHED_FILE_V2 = BASE / "circuits_unified_plus_aggressive_tier2_enriched.jsonl"
assert BROAD_ENRICHED_FILE_V2.exists(), f"Missing input file: {BROAD_ENRICHED_FILE_V2}"


def write_jsonl_any(entries, path):
    with open(path, "w", encoding="utf-8") as f:
        for entry in entries:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def core_rejection_reason_v2(entry):
    meta = entry.get("metadata", {})

    if meta.get("validation_status") != "validated":
        return "not_validated"
    if meta.get("extraction_confidence") != "high":
        return "not_high_confidence"
    if meta.get("contains_demo_scaffolding") is True:
        return "demo_scaffolding"
    if meta.get("cleanup_candidate") is True:
        return "cleanup_candidate"
    if (meta.get("code_lines") or 0) < 5:
        return "too_few_code_lines"
    if (meta.get("gate_count") or 0) < 2:
        return "too_few_gates"
    if meta.get("retrieval_strategy") == "empirical_promoted_repo":
        return "empirical_strategy_excluded"

    return None

broad_enriched_entries_v2 = load_jsonl_any(BROAD_ENRICHED_FILE_V2)

core_entries_v2 = []
tier2_entries_v2 = []
core_rejection_counts_v2 = Counter()
core_strategy_counts_v2 = Counter()
tier2_strategy_counts_v2 = Counter()

for entry in broad_enriched_entries_v2:
    meta = entry.get("metadata", {})
    strategy = meta.get("retrieval_strategy") or meta.get("source") or "<missing>"
    reason = core_rejection_reason_v2(entry)

    if reason is None:
        core_entries_v2.append(entry)
        core_strategy_counts_v2[strategy] += 1
    else:
        tier2_entries_v2.append(entry)
        tier2_strategy_counts_v2[strategy] += 1
        core_rejection_counts_v2[reason] += 1

write_jsonl_any(core_entries_v2, CORE_ENRICHED_FILE_V2)
write_jsonl_any(tier2_entries_v2, TIER2_ENRICHED_FILE_V2)

print("=" * 55)
print("CORE / TIER 2 SPLIT COMPLETE")
print("=" * 55)
print(f"Broad enriched total : {len(broad_enriched_entries_v2):,}")
print(f"Core benchmark set   : {len(core_entries_v2):,}")
print(f"Tier 2 set           : {len(tier2_entries_v2):,}")

print("\nCore strategy counts:")
for k, v in core_strategy_counts_v2.most_common():
    print(f"  {k}: {v:,}")

print("\nTier 2 strategy counts:")
for k, v in tier2_strategy_counts_v2.most_common():
    print(f"  {k}: {v:,}")

print("\nTop core rejection reasons:")
for k, v in core_rejection_counts_v2.most_common():
    print(f"  {k}: {v:,}")

print(f"\nCore file written to : {display_path(CORE_ENRICHED_FILE_V2)}")
print(f"Tier 2 file written to: {display_path(TIER2_ENRICHED_FILE_V2)}")


# Phase 3 High-Yield Recall Expansion

Phase 3 is the final acquisition expansion used in the active rebuild. Unlike Phase 2, it is intentionally selective in scope. Its purpose is to determine whether the existing pipeline is already near saturation and to recover only those remaining circuits that can be reached through especially high-signal search and recovery strategies.

The phase therefore avoids another broad empirical repository sweep and omits unconstrained owner-wide expansion. Instead, it concentrates on three controlled tests:

- re-sweeping a small set of trusted repositories to measure saturation
- targeting under-covered Qiskit construction idioms through a refined search-query pack
- optionally extending coverage to GitHub Gists for completeness

Phase 3 is the stage that yields the final merged raw acquisition pool used by the corrected benchmark workflow.


## Cell 20 — Phase 3 Configuration and Isolated State

This cell defines the output files, processed-state files, manifest path, counters, and retrieval labels specific to the high-yield Phase 3 pass. By isolating these artifacts from both baseline and Phase 2 outputs, the notebook preserves the ability to attribute later gains to a specific recall-expansion stage.

**Principal variables introduced in the code cell**

- `PHASE3_LABEL`, `RETRIEVAL_MODE_V3`, `RUN_ID_V3`: identifiers used to track Phase 3 provenance
- `OUTPUT_FILE_V3`, `PROCESSED_FILE_V3`, `MERGED_FILE_V3`, `MANIFEST_FILE_V3`: Phase 3 artifacts
- `PHASE3_TRUSTED_RESWEEP_REPOS`, `PHASE3_SEARCH_QUERIES_V2`, `PHASE3_NOTEBOOK_QUERY_PACK`: Phase 3 source lists

**Principal helper functions defined in the code cell**

- `repo_rel_display_v3()`: publication-safe relative-path formatter for Phase 3 status output
- `write_manifest_v3()`: writes the Phase 3 manifest


In [ ]:
# Cell 20 — Phase 3 high-yield config, manifest, and isolated state
from collections import Counter
from pathlib import Path

PHASE3_LABEL = "aggressive_v2_high_yield"
RETRIEVAL_MODE_V3 = "aggressive"
RUN_ID_V3 = f"{PHASE3_LABEL}_{SCRAPE_DATE}"

OUTPUT_FILE_V3 = BASE / "circuits_unified_phase3.jsonl"
PROCESSED_FILE_V3 = BASE / "circuits_unified_phase3_processed.txt"
MERGED_FILE_V3 = BASE / "circuits_unified_plus_phase2_plus_phase3.jsonl"
MANIFEST_FILE_V3 = BASE / "circuits_unified_phase3_manifest.json"

MAX_FILE_SIZE_BYTES_V3 = 1_500_000
INCLUDE_DOC_PATHS_V3 = True
SEARCH_PAGE_LIMIT_V3 = 8

SKIP_PATH_SEGMENTS_V3 = (
    "__pycache__", ".egg-info", "node_modules", "build/", "dist/", ".tox/",
)
PREFERRED_PATH_HINTS_V3 = (
    "examples/", "example/", "notebooks/", "tutorial", "demo", "docs/",
)

# Trusted, tutorial-heavy or bridge-style repos worth a cleaner third-pass sweep.
PHASE3_TRUSTED_RESWEEP_REPOS = [
    ("Qiskit", "documentation"),
    ("PennyLaneAI", "pennylane-qiskit"),
    ("Quantinuum", "pytket-qiskit"),
    ("sethuquantum", "LearnQuantum"),
    ("AayushSarkar", "Qiskit-Experiment-Hub"),
    ("AIComputing101", "quantum-computing-101"),
]

# High-signal indirect-construction patterns that Phase 2 may still under-recover.
PHASE3_SEARCH_QUERIES_V2 = list(dict.fromkeys([
    "from qiskit.circuit.library import NLocal language:python",
    "from qiskit.circuit.library import QAOAAnsatz language:python",
    "from qiskit.circuit.library import ExcitationPreserving language:python",
    "from qiskit.circuit.library import GroverOperator language:python",
    "from qiskit.circuit.library import PhaseEstimation language:python",
    "from qiskit.circuit.library import GraphState language:python",
    "from qiskit.circuit.library import UCCSD language:python",
    "QuantumCircuit.from_qasm_str( qiskit language:python",
    ".to_instruction( qiskit language:python",
    ".control( qiskit language:python",
    "from qiskit_algorithms language:python QuantumCircuit",
    "from qiskit_machine_learning language:python QuantumCircuit",
]))

# Notebook-heavy pass focused on tutorial and example recovery.
PHASE3_NOTEBOOK_QUERY_PACK = list(dict.fromkeys([
    "QuantumCircuit qiskit language:jupyter-notebook path:notebooks",
    "QuantumCircuit qiskit language:jupyter-notebook path:tutorial",
    "QuantumCircuit qiskit language:jupyter-notebook path:examples",
    "from qiskit.circuit.library language:jupyter-notebook",
    "measure_all() qiskit language:jupyter-notebook",
]))

if "REPO_ROOT" in globals():
    REPO_ROOT_V3 = Path(REPO_ROOT)
else:
    REPO_ROOT_V3 = next(
        (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists()),
        Path.cwd(),
    )

def repo_rel_display_v3(path):
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT_V3))
    except Exception:
        try:
            return str(Path(path))
        except Exception:
            return str(path)

def write_manifest_v3(extra=None):
    def _repo_rel(path):
        try:
            return str(Path(path).resolve().relative_to(REPO_ROOT_V3))
        except Exception:
            return str(path)

    manifest = {
        "phase": 3,
        "label": PHASE3_LABEL,
        "retrieval_mode": RETRIEVAL_MODE_V3,
        "retrieval_run_id": RUN_ID_V3,
        "scrape_date": SCRAPE_DATE,
        "baseline_output_file": _repo_rel(OUTPUT_FILE),
        "phase2_output_file": _repo_rel(OUTPUT_FILE_V2),
        "phase3_output_file": _repo_rel(OUTPUT_FILE_V3),
        "merged_output_file": _repo_rel(MERGED_FILE_V3),
        "processed_file_v3": _repo_rel(PROCESSED_FILE_V3),
        "max_file_size_bytes_v3": MAX_FILE_SIZE_BYTES_V3,
        "include_doc_paths_v3": INCLUDE_DOC_PATHS_V3,
        "trusted_resweep_repos_v3": PHASE3_TRUSTED_RESWEEP_REPOS,
        "search_queries_v3": PHASE3_SEARCH_QUERIES_V2,
        "notebook_queries_v3": PHASE3_NOTEBOOK_QUERY_PACK,
        "phase3_variant": "high_yield_only",
    }
    if extra:
        manifest.update(extra)
    with open(MANIFEST_FILE_V3, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

processed_v3 = load_processed(PROCESSED_FILE_V3)
seen_hashes_v3 = (
    load_seen_hashes(OUTPUT_FILE)
    | load_seen_hashes(OUTPUT_FILE_V2)
    | load_seen_hashes(OUTPUT_FILE_V3)
)

write_manifest_v3()

print(f"Phase 3 label       : {PHASE3_LABEL}")
print(f"Phase 3 output      : {repo_rel_display_v3(OUTPUT_FILE_V3)}")
print(f"Phase 3 processed   : {repo_rel_display_v3(PROCESSED_FILE_V3)}")
print(f"Phase 3 merged file : {repo_rel_display_v3(MERGED_FILE_V3)}")
print(f"Combined seen hashes: {len(seen_hashes_v3):,}")

## Cell 21 — Phase 3 Helper Wrappers

This cell defines the Phase 3-specific metadata templates and helper wrappers. These functions preserve consistency with earlier acquisition stages while ensuring that Phase 3 entries carry distinct provenance information for downstream reporting.

**Principal functions defined in the code cell**

- `metadata_template_v3()`: constructs Phase 3 retrieval-aware metadata payloads
- `get_repo_code_files_v3()`: enumerates candidate files for Phase 3 repository sweeps
- `fetch_file_circuits_v3()`: retrieves and extracts circuits from a single file
- `process_repo_v3()`: repository-level processing wrapper for Phase 3
- `run_search_queries_v3()`: shared query runner used by the high-signal and notebook-heavy search packs


In [ ]:
# Cell 21 — Phase 3 high-yield helper wrappers
def metadata_template_v3(
    owner, repo, path, file_url, file_sha, ext,
    start_line, end_line, code, source_tag, retrieval_strategy
):
    meta = {k: None for k in SCHEMA_NULL_FIELDS}
    github_anchor = file_url if start_line is None else f"{file_url}#L{start_line}-L{end_line}"
    meta.update({
        "original_url": file_url,
        "file_path": path,
        "source": source_tag,
        "language": "jupyter" if ext == "ipynb" else "python",
        "circuit_hash": hashlib.md5(code.strip().encode("utf-8")).hexdigest(),
        "hash": file_sha,
        "start_line": start_line,
        "end_line": end_line,
        "github_anchor": github_anchor,
        "repo_owner": owner,
        "repo_name": repo,
        "scrape_date": SCRAPE_DATE,
        "code_lines": len([l for l in code.splitlines() if l.strip()]),
        "retrieval_mode": RETRIEVAL_MODE_V3,
        "retrieval_strategy": retrieval_strategy,
        "retrieval_run_id": RUN_ID_V3,
    })
    return meta

def get_repo_code_files_v3(session, owner, repo, branch):
    tree_url = f"{API_BASE}/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    data = api_get(session, tree_url)
    if not data:
        return []

    files = []
    for item in data.get("tree", []):
        path = item.get("path", "")
        lower = path.lower()
        if item.get("type") != "blob":
            continue
        if not lower.endswith((".py", ".ipynb")):
            continue
        if item.get("size", 0) > MAX_FILE_SIZE_BYTES_V3:
            continue
        if any(seg.lower() in lower for seg in SKIP_PATH_SEGMENTS_V3):
            continue
        if (not INCLUDE_DOC_PATHS_V3) and "docs/" in lower:
            continue
        files.append(path)

    files.sort(
        key=lambda p: (
            0 if any(h in p.lower() for h in PREFERRED_PATH_HINTS_V3) else 1,
            p.lower(),
        )
    )
    return files

def fetch_file_circuits_v3(
    session, owner, repo, path, branch,
    source_tag, retrieval_strategy,
    seen_hashes, processed, processed_file
):
    file_url = f"https://github.com/{owner}/{repo}/blob/{branch}/{path}"
    if file_url in processed:
        return []

    api_url = f"{API_BASE}/repos/{owner}/{repo}/contents/{path}"
    data = api_get(session, api_url, params={"ref": branch})

    mark_processed_to(file_url, processed_file)
    processed.add(file_url)

    if not data or not isinstance(data, dict):
        return []
    if data.get("size", 0) > MAX_FILE_SIZE_BYTES_V3:
        return []

    raw_content = data.get("content", "")
    if not raw_content:
        return []

    decoded = base64.b64decode(raw_content).decode("utf-8", errors="replace")
    file_sha = data.get("sha", "")
    ext = path.lower().split(".")[-1]

    tuples = (
        extract_circuits_notebook_aggressive(decoded)
        if ext == "ipynb"
        else extract_circuits_python_aggressive(decoded)
    )

    entries = []
    for code, start_line, end_line in tuples:
        ch = hashlib.md5(code.strip().encode("utf-8")).hexdigest()
        if ch in seen_hashes:
            continue
        seen_hashes.add(ch)

        entries.append({
            "input": "",
            "output": code,
            "openqasm3_code": None,
            "metadata": metadata_template_v3(
                owner, repo, path, file_url, file_sha, ext,
                start_line, end_line, code, source_tag, retrieval_strategy
            ),
        })

    return entries

def process_repo_v3(
    session, owner, repo, source_tag, retrieval_strategy,
    seen_hashes, processed, processed_file, output_file
):
    branch = get_default_branch(session, owner, repo)
    files = get_repo_code_files_v3(session, owner, repo, branch)
    count = 0

    for path in files:
        for entry in fetch_file_circuits_v3(
            session, owner, repo, path, branch,
            source_tag, retrieval_strategy,
            seen_hashes, processed, processed_file,
        ):
            append_jsonl_to(entry, output_file)
            count += 1

    return count

def run_search_queries_v3(query_list, source_tag, retrieval_strategy):
    total = 0
    per_query = {}

    for qi, query in enumerate(query_list, 1):
        print(f"[{qi}/{len(query_list)}] {query}")
        page = 1
        query_total = 0

        while page <= SEARCH_PAGE_LIMIT_V3:
            data = api_get(
                session,
                f"{API_BASE}/search/code",
                params={"q": query, "per_page": 100, "page": page},
                sleep=SEARCH_SLEEP,
            )
            if not data:
                break

            items = data.get("items", [])
            if not items:
                break

            for item in items:
                repo_info = item.get("repository", {})
                owner = repo_info.get("owner", {}).get("login", "")
                repo = repo_info.get("name", "")
                path = item.get("path", "")
                if not (owner and repo and path):
                    continue

                branch = get_default_branch(session, owner, repo)
                for entry in fetch_file_circuits_v3(
                    session, owner, repo, path, branch,
                    source_tag=source_tag,
                    retrieval_strategy=retrieval_strategy,
                    seen_hashes=seen_hashes_v3,
                    processed=processed_v3,
                    processed_file=PROCESSED_FILE_V3,
                ):
                    append_jsonl_to(entry, OUTPUT_FILE_V3)
                    query_total += 1
                    total += 1

            if len(items) < 100:
                break
            page += 1

        per_query[query] = query_total
        print(f"  -> {query_total} circuits (running total={total:,})")

    return total, per_query

print("Phase 3 high-yield helpers ready.")

## Cell 22 — Strategy D: Trusted Repository Re-sweeps

This strategy revisits a small set of trusted tutorial and documentation repositories. Its primary purpose is not necessarily to add many new circuits, but to test whether these repositories are already saturated under the current extractor.

A near-zero yield here is therefore informative rather than disappointing: it constitutes evidence that the existing pipeline had already exhausted much of the high-trust repository surface.

**Principal counters used in the code cell**

- `total_resweep_v3`: cumulative number of circuits added by trusted re-sweeps
- `resweep_counts_v3`: per-repository yield summary for the trusted re-sweep stage


In [ ]:
# Cell 22 — Strategy D: trusted tutorial/documentation re-sweeps
total_resweep_v3 = 0
resweep_counts_v3 = {}

for i, (owner, repo) in enumerate(PHASE3_TRUSTED_RESWEEP_REPOS, 1):
    n = process_repo_v3(
        session, owner, repo,
        source_tag="trusted_resweep_v3",
        retrieval_strategy="trusted_resweep",
        seen_hashes=seen_hashes_v3,
        processed=processed_v3,
        processed_file=PROCESSED_FILE_V3,
        output_file=OUTPUT_FILE_V3,
    )
    resweep_counts_v3[f"{owner}/{repo}"] = n
    total_resweep_v3 += n
    print(f"[{i}/{len(PHASE3_TRUSTED_RESWEEP_REPOS)}] {owner}/{repo} -> +{n}")

print(f"\nTrusted re-sweep total: {total_resweep_v3:,}")

## Cell 23 — Strategy E: High-Signal Expanded Search v2

This strategy executes the refined Phase 3 search-query pack. The queries emphasize Qiskit construction idioms that were comparatively underrepresented in earlier phases, including circuit-library abstractions, representation-conversion pathways, and reusable instruction wrappers.

Empirically, this strategy is the principal driver of Phase 3 yield and is therefore central to the interpretation of the final recall-expansion stage.

**Principal outputs produced by the code cell**

- `total_search_v3`: total number of circuits added by the refined high-signal query pack
- `search_query_counts_v3`: per-query yield summary for the high-signal search stage


In [ ]:
# Cell 23 — Strategy E: high-signal expanded search v2
total_search_v3, search_query_counts_v3 = run_search_queries_v3(
    PHASE3_SEARCH_QUERIES_V2,
    source_tag="search_v3",
    retrieval_strategy="expanded_search_v2",
)

print(f"\nExpanded search v2 total: {total_search_v3:,}")

## Cell 24 — Strategy F: Notebook-Heavy Search Pack

This strategy tests whether GitHub-indexed notebooks still contribute incremental circuit yield after the baseline and aggressive phases. Its value is partly additive and partly diagnostic: a low-yield result supports the interpretation that notebook-oriented retrieval was already near saturation.

**Principal outputs produced by the code cell**

- `total_notebook_v3`: total number of circuits added by the notebook-heavy query pack
- `notebook_query_counts_v3`: per-query yield summary for notebook-oriented search


In [ ]:
# Cell 24 — Strategy F: notebook-heavy query pack
total_notebook_v3, notebook_query_counts_v3 = run_search_queries_v3(
    PHASE3_NOTEBOOK_QUERY_PACK,
    source_tag="search_notebook_v3",
    retrieval_strategy="notebook_search_pack",
)

print(f"\nNotebook-heavy search total: {total_notebook_v3:,}")

## Cell 25 — Optional Gist Helper Definitions

This cell defines the helper functions used for the optional GitHub Gist recovery pass. The gist pass is deliberately limited in scope and is intended primarily as a completeness check over an additional GitHub surface rather than as a major source of circuits.

**Principal functions defined in the code cell**

- `extract_gist_id()`: normalizes gist URLs into gist identifiers
- `load_gist_urls()`: loads gist URLs from the repository source list
- `fetch_gist_circuits_v3()`: retrieves and extracts circuit entries from a GitHub Gist


In [ ]:
# Cell 25 Optional Phase 3 gist pass
def extract_gist_id(url: str):
    url = url.strip()
    if "gist.github.com/" not in url:
        return None
    parts = [p for p in url.split("/") if p]
    if not parts:
        return None
    return parts[-1].split("#")[0]

def load_gist_urls(path):
    urls = []
    if not Path(path).exists():
        return urls
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if "gist.github.com/" in line:
                urls.append(line)
    return urls

def fetch_gist_circuits_v3(session, gist_url, seen_hashes, processed, processed_file):
    if gist_url in processed:
        return []

    gist_id = extract_gist_id(gist_url)
    if not gist_id:
        return []

    data = api_get(session, f"{API_BASE}/gists/{gist_id}")
    mark_processed_to(gist_url, processed_file)
    processed.add(gist_url)

    if not data:
        return []

    owner = ((data.get("owner") or {}).get("login")) or "gist"
    repo_name = f"gist-{gist_id}"
    html_url = data.get("html_url", gist_url)

    entries = []
    for filename, info in (data.get("files") or {}).items():
        lower = filename.lower()
        if not lower.endswith((".py", ".ipynb")):
            continue

        content = info.get("content") or ""
        if not content:
            continue

        ext = lower.split(".")[-1]
        tuples = (
            extract_circuits_notebook_aggressive(content)
            if ext == "ipynb"
            else extract_circuits_python_aggressive(content)
        )

        for code, start_line, end_line in tuples:
            ch = hashlib.md5(code.strip().encode("utf-8")).hexdigest()
            if ch in seen_hashes:
                continue
            seen_hashes.add(ch)

            github_anchor = html_url if start_line is None else f"{html_url}#file-{filename.replace('.', '-')}"
            meta = {k: None for k in SCHEMA_NULL_FIELDS}
            meta.update({
                "original_url": html_url,
                "file_path": filename,
                "source": "gist_v3",
                "language": "jupyter" if ext == "ipynb" else "python",
                "circuit_hash": ch,
                "hash": gist_id,
                "start_line": start_line,
                "end_line": end_line,
                "github_anchor": github_anchor,
                "repo_owner": owner,
                "repo_name": repo_name,
                "scrape_date": SCRAPE_DATE,
                "code_lines": len([l for l in code.splitlines() if l.strip()]),
                "retrieval_mode": RETRIEVAL_MODE_V3,
                "retrieval_strategy": "gist",
                "retrieval_run_id": RUN_ID_V3,
            })

            entries.append({
                "input": "",
                "output": code,
                "openqasm3_code": None,
                "metadata": meta,
            })
    return entries


### Gist URL Loading and Optional Recovery Pass

This cell loads any `gist.github.com` entries from the curated URL list and executes the optional Phase 3 gist pass. The step is intentionally lightweight: it broadens GitHub surface coverage without materially changing the overall recall strategy.

**Principal variables used in the code cell**

- `gist_file` and `GIST_URLS_V3`: location of the curated URL list and the subset of discovered gist URLs
- `total_gist_v3`: total number of circuits recovered from the optional gist pass
- `entries`: per-gist recovery results appended to the Phase 3 output file


In [ ]:
gist_file = REPO_ROOT_V3 / "github_urls.txt"
GIST_URLS_V3 = load_gist_urls(gist_file)
total_gist_v3 = 0

if not GIST_URLS_V3:
    print("No gist URLs were found in the curated URL list; the optional gist pass is skipped.")
else:
    print(f"Discovered {len(GIST_URLS_V3):,} gist URL(s) for optional recovery.")
    for i, gist_url in enumerate(GIST_URLS_V3, 1):
        entries = fetch_gist_circuits_v3(
            session,
            gist_url,
            seen_hashes=seen_hashes_v3,
            processed=processed_v3,
            processed_file=PROCESSED_FILE_V3,
        ) or []

        for entry in entries:
            append_jsonl_to(entry, OUTPUT_FILE_V3)
            total_gist_v3 += 1

        print(f"[{i}/{len(GIST_URLS_V3)}] gist -> +{len(entries)}")

print(f"\nGist total: {total_gist_v3:,}")


## Cell 26 — Final Merge and Retrieval-Metadata Backfill

This merge stage combines the baseline, Phase 2, and Phase 3 outputs into the final raw acquisition pool while normalizing and backfilling retrieval metadata across all entries. The resulting merged file is the canonical raw artifact for the active rebuild.

This is the point at which acquisition ends and post-acquisition processing begins.

**Principal helper functions defined in the code cell**

- `normalize_retrieval_metadata_v3()`: ensures every merged entry carries explicit retrieval provenance
- `merge_unique_by_circuit_hash_v3()`: merges baseline, Phase 2, and Phase 3 outputs while deduplicating by circuit hash

**Principal merged-state variables created in the code cell**

- `merged_total_v3`: total size of the final merged raw acquisition pool
- `merged_mode_counts_v3` and `merged_strategy_counts_v3`: retrieval-provenance summaries of the final merged raw pool


In [ ]:
# Cell 26 — Merge baseline + Phase 2 + Phase 3 and backfill retrieval metadata
def normalize_retrieval_metadata_v3(entry, default_mode, default_run_id):
    entry = dict(entry)
    entry["metadata"] = dict(entry.get("metadata", {}))
    meta = entry["metadata"]

    if meta.get("retrieval_mode") in (None, ""):
        meta["retrieval_mode"] = default_mode
    if meta.get("retrieval_strategy") in (None, ""):
        meta["retrieval_strategy"] = meta.get("source", "unknown")
    if meta.get("retrieval_run_id") in (None, ""):
        meta["retrieval_run_id"] = default_run_id

    return entry

def merge_unique_by_circuit_hash_v3(sources, output_path):
    seen = set()
    total = 0

    with open(output_path, "w", encoding="utf-8") as out:
        for path, default_mode, default_run_id in sources:
            if not path.exists():
                continue

            with open(path, encoding="utf-8") as f:
                for line in f:
                    if not line.strip():
                        continue

                    entry = normalize_retrieval_metadata_v3(
                        json.loads(line),
                        default_mode=default_mode,
                        default_run_id=default_run_id,
                    )
                    ch = entry.get("metadata", {}).get("circuit_hash", "")
                    if ch and ch in seen:
                        continue
                    if ch:
                        seen.add(ch)

                    out.write(json.dumps(entry, ensure_ascii=False) + "\n")
                    total += 1

    return total

merged_total_v3 = merge_unique_by_circuit_hash_v3(
    [
        (OUTPUT_FILE, "baseline", "baseline_legacy"),
        (OUTPUT_FILE_V2, "aggressive", globals().get("RUN_ID_V2", "aggressive_v1_legacy")),
        (OUTPUT_FILE_V3, "aggressive", RUN_ID_V3),
    ],
    MERGED_FILE_V3,
)

baseline_count_v3 = len(load_jsonl_any(OUTPUT_FILE))
phase2_count_v3 = len(load_jsonl_any(OUTPUT_FILE_V2))
phase3_count_v3 = len(load_jsonl_any(OUTPUT_FILE_V3))
merged_entries_v3 = load_jsonl_any(MERGED_FILE_V3)

merged_mode_counts_v3 = Counter(
    e.get("metadata", {}).get("retrieval_mode", "<missing>")
    for e in merged_entries_v3
)
merged_strategy_counts_v3 = Counter(
    e.get("metadata", {}).get("retrieval_strategy", "<missing>")
    for e in merged_entries_v3
)

print("=" * 55)
print("PHASE 3 COMPLETE")
print("=" * 55)
print(f"Baseline raw pool           : {baseline_count_v3:,}")
print(f"Phase 2 additions           : {phase2_count_v3:,}")
print(f"Phase 3 additions           : {phase3_count_v3:,}")
print(f"Merged unique circuits      : {merged_total_v3:,}")
print(f"Trusted re-sweeps added     : {total_resweep_v3:,}")
print(f"Expanded search v2 added    : {total_search_v3:,}")
print(f"Notebook query pack added   : {total_notebook_v3:,}")

print("\nMerged retrieval_mode counts:")
for k, v in merged_mode_counts_v3.items():
    print(f"  {k}: {v:,}")

print("\nMerged retrieval_strategy counts:")
for k, v in merged_strategy_counts_v3.most_common():
    print(f"  {k}: {v:,}")

write_manifest_v3({
    "baseline_count": baseline_count_v3,
    "phase2_count": phase2_count_v3,
    "phase3_count": phase3_count_v3,
    "merged_count": merged_total_v3,
    "trusted_resweep_total": total_resweep_v3,
    "search_v2_total": total_search_v3,
    "notebook_search_total": total_notebook_v3,
    "merged_retrieval_mode_counts": dict(merged_mode_counts_v3),
    "merged_retrieval_strategy_counts": dict(merged_strategy_counts_v3),
    "phase3_variant": "high_yield_only",
})

print(f"\nMerged file written to: {repo_rel_display_v3(MERGED_FILE_V3)}")
print(f"Manifest written to   : {repo_rel_display_v3(MANIFEST_FILE_V3)}")

# Phase 3 Post-Processing

The final section reruns enrichment, reporting, and benchmark tiering over the merged Phase 3 corpus using the corrected `materialized_circuit` logic. Only the preferred publication-safe execution path is retained below.


## Post-Processing A — Python 3.11 Resolver

This cell resolves a Python 3.11 interpreter for Qiskit-dependent reruns. The notebook kernel may differ from the interpreter required by the enrichment script, so the resolver keeps the environment requirement explicit without hardcoding a personal absolute path into the notebook source.

**Principal variables used in the code cell**

- `ROOT`: repository root resolved at runtime
- `python_candidates`: candidate Python 3.11 locations examined by the resolver
- `PYTHON311`: executable selected for subsequent subprocess-based reruns


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())

python_candidates = [
    Path(os.environ.get("PYTHON311", "")),
    Path.home() / "AppData/Local/Programs/Python/Python311/python.exe",
    Path(os.environ.get("LOCALAPPDATA", "")) / "Programs/Python/Python311/python.exe",
    Path(sys.executable),
]

PYTHON311 = next((p for p in python_candidates if str(p) and p.exists()), None)
assert PYTHON311 is not None, "Python 3.11 interpreter not found. Set PYTHON311 or install Python 3.11."

print("Python 3.11 interpreter resolved:", True)


## Post-Processing B — Preferred Subprocess Enrichment Rerun

This is the preferred rerun method for final Phase 3 enrichment. By using `subprocess.run(...)` rather than a bare `!python` call, the cell avoids Jupyter echoing a `CompletedProcess(...)` command containing local absolute paths. It is therefore the appropriate rerun path for publication-facing notebook execution.

**Principal variables used in the code cell**

- `phase3_raw` and `phase3_enriched`: final merged raw input and enriched output files
- `script`: path to `enrich_raw_circuits.py`
- `result`: subprocess completion object used to confirm successful execution


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())

python_candidates = [
    Path(os.environ["PYTHON311"]).expanduser() if os.environ.get("PYTHON311") else None,
    Path(os.environ["LOCALAPPDATA"]) / "Programs/Python/Python311/python.exe" if os.environ.get("LOCALAPPDATA") else None,
    Path.home() / "AppData/Local/Programs/Python/Python311/python.exe",
    Path(sys.executable),
]

python_candidates = [p for p in python_candidates if p is not None and p.is_file() and p.name.lower() == "python.exe"]

assert python_candidates, "No valid Python 3.11 executable found."

PYTHON311 = python_candidates[0]

print("python311 executable found:", PYTHON311.is_file())
result = subprocess.run([str(PYTHON311), "--version"], check=True, capture_output=True, text=True)
print("python version check passed:", result.stdout.strip() or result.stderr.strip())


### Post-Processing B1 — Execute the Enrichment Rerun

This cell executes the corrected Phase 3 enrichment rerun using the Python 3.11 interpreter resolved in the preceding setup cell. It is kept separate from the resolver so that interpreter validation and dataset regeneration remain distinct steps in the notebook narrative.


In [ ]:
import subprocess

phase3_raw = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3.jsonl"
phase3_enriched = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
script = ROOT / "PQID/scripts/enrich_raw_circuits.py"

print("raw exists:", phase3_raw.exists())
print("python311 ready:", True)

result = subprocess.run(
    [
        str(PYTHON311),
        str(script),
        "--input-file",
        str(phase3_raw),
        "--output-file",
        str(phase3_enriched),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("enrichment subprocess completed with return code:", result.returncode)

## Post-Processing C — Preferred Subprocess Extraction Audit

This is the preferred rerun method for generating the corrected Phase 3 extraction-quality report. It preserves explicit notebook-level orchestration while reducing the risk of path leakage in stored outputs.

**Principal variables used in the code cell**

- `phase3_enriched`: corrected enriched Phase 3 pool used as report input
- `report_file` and `samples_file`: extraction-audit outputs
- `result`: subprocess completion object used to confirm successful execution


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
phase3_enriched = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
report_file = ROOT / "PQID/data/processed/extraction_quality_report_phase3.md"
samples_file = ROOT / "PQID/data/processed/extraction_quality_samples_phase3.jsonl"

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "PQID/scripts/report_extraction_quality.py"),
        "--input-file",
        str(phase3_enriched),
        "--report-file",
        str(report_file),
        "--samples-file",
        str(samples_file),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("report subprocess completed with return code:", result.returncode)

## Post-Processing D — Preferred Subprocess Strict Tiering

This is the preferred rerun method for generating the strict benchmark core and strict Tier 2 split from the corrected enriched pool. It should be used when regenerating the public strict-core benchmark artifact.

**Principal variables used in the code cell**

- `phase3_core`, `phase3_tier2`, and `phase3_report`: strict benchmark outputs regenerated by the preferred subprocess path
- `result`: subprocess completion object used to confirm successful execution


In [ ]:
phase3_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_enriched.jsonl"
phase3_tier2 = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_enriched.jsonl"
phase3_report = ROOT / "PQID/data/processed/benchmark_tiering_report_phase3.md"

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "PQID/scripts/filter_benchmark_and_tier2.py"),
        "--input-file",
        str(phase3_enriched),
        "--core-output-file",
        str(phase3_core),
        "--tier2-output-file",
        str(phase3_tier2),
        "--report-file",
        str(phase3_report),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("benchmark tiering subprocess completed with return code:", result.returncode)

## Post-Processing E — Preferred Subprocess Extended Tiering

This is the preferred rerun method for generating the extended benchmark core with validated high-quality empirical entries included. The resulting extended export is the recommended source for the next stage of seed generation.

**Principal variables used in the code cell**

- `phase3_core_ext`, `phase3_tier2_ext`, and `phase3_report_ext`: extended benchmark outputs regenerated by the preferred subprocess path
- `result`: subprocess completion object used to confirm successful execution


In [ ]:
phase3_core_ext = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_enriched.jsonl"
phase3_tier2_ext = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_extended.jsonl"
phase3_report_ext = ROOT / "PQID/data/processed/benchmark_tiering_report_phase3_extended.md"

result =subprocess.run(
    [
        sys.executable,
        str(ROOT / "PQID/scripts/filter_benchmark_and_tier2.py"),
        "--input-file",
        str(phase3_enriched),
        "--core-output-file",
        str(phase3_core_ext),
        "--tier2-output-file",
        str(phase3_tier2_ext),
        "--report-file",
        str(phase3_report_ext),
        "--include-empirical-in-core",
    ],
    check=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("extended benchmark tiering subprocess completed with return code:", result.returncode)

## Post-Processing F — Dominant Repository Audit

This audit cell investigates whether the unexpectedly large contribution from a single repository reflects accidental duplication or a genuinely large subcorpus. The analysis is performed directly on the uncleaned extended benchmark export so that the benchmark-design problem can be demonstrated before any corrective filtering is applied.

**Principal variables used in the code cell**

- `dominant_repo_owner` and `dominant_repo_name`: repository identifiers used to isolate the suspect source
- `uncleaned_extended_core`: extended benchmark export prior to mutation-path cleaning
- `retrieval_counts` and `score_counts`: distributions used to show how the dominant repository enters the benchmark
- `unique_circuit_hashes`, `unique_source_hashes`, `unique_paths`, and `unique_urls`: uniqueness diagnostics used to rule out naive duplication


In [ ]:
import json
from collections import Counter

dominant_repo_owner = "Ahmik-Virani"
dominant_repo_name = "Differentiating-Quantum-Bug-From-Noise-Statistical-Approach"
uncleaned_extended_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_enriched.jsonl"

total = 0
unique_circuit_hashes = set()
unique_source_hashes = set()
unique_paths = set()
unique_urls = set()
retrieval_counts = Counter()
score_counts = Counter()

with open(uncleaned_extended_core, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        meta = obj.get("metadata", {})
        if meta.get("repo_owner") == dominant_repo_owner and meta.get("repo_name") == dominant_repo_name:
            total += 1
            if meta.get("circuit_hash"):
                unique_circuit_hashes.add(meta["circuit_hash"])
            if meta.get("hash"):
                unique_source_hashes.add(meta["hash"])
            if meta.get("file_path"):
                unique_paths.add(meta["file_path"])
            if meta.get("original_url"):
                unique_urls.add(meta["original_url"])
            retrieval_counts[meta.get("retrieval_strategy")] += 1
            score_counts[meta.get("benchmark_checks_passed")] += 1

print("Dominant repository audit")
print("=" * 40)
print(f"repository                    : {dominant_repo_owner}/{dominant_repo_name}")
print(f"entries in extended core       : {total:,}")
print(f"unique circuit_hash            : {len(unique_circuit_hashes):,}")
print(f"unique source hash             : {len(unique_source_hashes):,}")
print(f"unique file_path               : {len(unique_paths):,}")
print(f"unique original_url            : {len(unique_urls):,}")
print()
print("retrieval strategy distribution:")
for key, value in retrieval_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("benchmark suitability score distribution within the repository:")
for passed in sorted(score_counts):
    print(f"  {passed}/7: {score_counts[passed]:,}")


## Post-Processing G — Mutation-Path Impact Audit

This cell determines whether the dominant repository’s benchmark contribution arises mainly from mutation-suite files. It then quantifies how much of the original strict and extended benchmark exports would be removed if mutation-suite paths were excluded. This provides the empirical justification for the cleaning round implemented in the following cells.

**Principal variables used in the code cell**

- `mutation_markers`: path fragments used to identify mutation-suite files
- `dominant_repo_mutants` and `dominant_repo_non_mutants`: path-level counts within the dominant repository
- `strict_mutation_paths` and `extended_mutation_paths`: benchmark-level mutation counts before cleaning
- `strict_core_cleaned_count` and `extended_core_cleaned_count`: post-cleaning benchmark sizes used to quantify the effect of the filter


In [ ]:
import json

mutation_markers = ("/mutants/", "mutants_of_")
uncleaned_strict_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_enriched.jsonl"
cleaned_strict_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_cleaned_enriched.jsonl"
cleaned_extended_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_cleaned_enriched.jsonl"

def normalized_path(meta):
    return (meta.get("file_path") or meta.get("original_url") or "").replace("\\", "/").lower()

def is_mutation_path(meta):
    path = normalized_path(meta)
    return any(marker in path for marker in mutation_markers)

dominant_repo_mutants = 0
dominant_repo_non_mutants = 0
with open(uncleaned_extended_core, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        meta = obj.get("metadata", {})
        if meta.get("repo_owner") == dominant_repo_owner and meta.get("repo_name") == dominant_repo_name:
            if is_mutation_path(meta):
                dominant_repo_mutants += 1
            else:
                dominant_repo_non_mutants += 1

def count_mutation_paths(path):
    total = 0
    mutation_count = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            total += 1
            meta = json.loads(line).get("metadata", {})
            if is_mutation_path(meta):
                mutation_count += 1
    return total, mutation_count

strict_total, strict_mutation_paths = count_mutation_paths(uncleaned_strict_core)
extended_total, extended_mutation_paths = count_mutation_paths(uncleaned_extended_core)
strict_cleaned_total, _ = count_mutation_paths(cleaned_strict_core)
extended_cleaned_total, _ = count_mutation_paths(cleaned_extended_core)

print("Mutation-path audit")
print("=" * 40)
print(f"dominant repo mutation-suite paths  : {dominant_repo_mutants:,}")
print(f"dominant repo non-mutation paths    : {dominant_repo_non_mutants:,}")
print()
print(f"strict core before cleaning         : {strict_total:,}")
print(f"strict-core mutation paths          : {strict_mutation_paths:,}")
print(f"strict core after cleaning          : {strict_cleaned_total:,}")
print()
print(f"extended core before cleaning       : {extended_total:,}")
print(f"extended-core mutation paths        : {extended_mutation_paths:,}")
print(f"extended core after cleaning        : {extended_cleaned_total:,}")


## Post-Processing H — License Composition After Cleaning

This audit cell documents how the mutation-path cleaning round changes the governance profile of the benchmark exports. It does not resolve the residual `no_license` issue, but it makes the remaining release constraints explicit and quantifiable.

**Principal variables used in the code cell**

- `strict_license_counts` and `extended_license_counts`: license-category distributions for the cleaned strict and cleaned extended benchmark exports
- `strict_no_license_repos` and `extended_no_license_repos`: top remaining no-license source repositories after cleaning


In [ ]:
from pathlib import Path
import json
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
cleaned_strict_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_cleaned_enriched.jsonl"
cleaned_extended_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_cleaned_enriched.jsonl"

def license_snapshot(path):
    license_counts = Counter()
    no_license_repos = Counter()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            meta = obj.get("metadata", {})
            category = meta.get("license_category") or "<missing>"
            license_counts[category] += 1
            if category == "no_license":
                no_license_repos[f"{meta.get('repo_owner')}/{meta.get('repo_name')}"] += 1
    return license_counts, no_license_repos

strict_license_counts, strict_no_license_repos = license_snapshot(cleaned_strict_core)
extended_license_counts, extended_no_license_repos = license_snapshot(cleaned_extended_core)

print("Cleaned strict-core license distribution:")
for key, value in strict_license_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("Top remaining no-license repos in cleaned strict core:")
for key, value in strict_no_license_repos.most_common(10):
    print(f"  {key}: {value:,}")
print()
print("Cleaned extended-core license distribution:")
for key, value in extended_license_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("Top remaining no-license repos in cleaned extended core:")
for key, value in extended_no_license_repos.most_common(10):
    print(f"  {key}: {value:,}")


## Post-Processing I — Cleaned Strict Tiering

This cleaning round regenerates the strict benchmark core after excluding mutation-suite file paths such as `*/Mutants/*`. The goal is to prevent a single synthetic mutation corpus from dominating the public benchmark definition.

**Principal variables used in the code cell**

- `phase3_core_cleaned`, `phase3_tier2_cleaned`, and `phase3_report_cleaned`: cleaned strict benchmark outputs
- `result`: subprocess completion object used to confirm successful execution


In [ ]:
import subprocess
import sys

phase3_enriched = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
phase3_core_cleaned = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_cleaned_enriched.jsonl"
phase3_tier2_cleaned = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_cleaned_enriched.jsonl"
phase3_report_cleaned = ROOT / "PQID/data/processed/benchmark_tiering_report_phase3_cleaned.md"

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "PQID/scripts/filter_benchmark_and_tier2_cleaned.py"),
        "--input-file",
        str(phase3_enriched),
        "--core-output-file",
        str(phase3_core_cleaned),
        "--tier2-output-file",
        str(phase3_tier2_cleaned),
        "--report-file",
        str(phase3_report_cleaned),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("cleaned strict benchmark tiering subprocess completed with return code:", result.returncode)

## Post-Processing J — Cleaned Extended Tiering

This cleaning round regenerates the extended benchmark core with empirical near-core entries retained unless they originate from mutation-suite paths. The resulting cleaned extended export should be treated as the benchmark-ready source for any public release or subsequent instruction-generation stage.

**Principal variables used in the code cell**

- `phase3_core_ext_cleaned`, `phase3_tier2_ext_cleaned`, and `phase3_report_ext_cleaned`: cleaned extended benchmark outputs
- `result`: subprocess completion object used to confirm successful execution


In [ ]:
import subprocess
import sys

phase3_enriched = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
phase3_core_ext_cleaned = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_cleaned_enriched.jsonl"
phase3_tier2_ext_cleaned = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_extended_cleaned.jsonl"
phase3_report_ext_cleaned = ROOT / "PQID/data/processed/benchmark_tiering_report_phase3_extended_cleaned.md"

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "PQID/scripts/filter_benchmark_and_tier2_cleaned.py"),
        "--input-file",
        str(phase3_enriched),
        "--core-output-file",
        str(phase3_core_ext_cleaned),
        "--tier2-output-file",
        str(phase3_tier2_ext_cleaned),
        "--report-file",
        str(phase3_report_ext_cleaned),
        "--include-empirical-in-core",
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("cleaned extended benchmark tiering subprocess completed with return code:", result.returncode)

## Post-Processing K — Repository Concentration After Cleaning

This cell evaluates whether the mutation-cleaned benchmark exports are sufficiently source-balanced to serve as benchmark artifacts. It reports top-repository share, top-5 and top-10 shares, and a simple Herfindahl-Hirschman-style concentration index for the cleaned strict and cleaned extended exports. It also computes a diagnostic scenario in which the dominant cleaned-extended repository (`backordinary/QDP-FSL`) is excluded, allowing the manuscript to distinguish between a balanced strict benchmark and a still-concentrated development pool.

**Principal variables used in the code cell**

- `strict_repo_summary` and `extended_repo_summary`: concentration summaries for the cleaned strict and cleaned extended benchmark exports
- `extended_without_qdp_fsl_summary`: counterfactual concentration summary used to measure the effect of the dominant remaining repository
- `summarize_repo_concentration()`: helper that computes repo-frequency counts, top-share statistics, and HHI-style concentration


In [ ]:
from pathlib import Path
import json
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
cleaned_strict_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_cleaned_enriched.jsonl"
cleaned_extended_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_cleaned_enriched.jsonl"

def summarize_repo_concentration(path, *, exclude_repo=None):
    repo_counts = Counter()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            meta = json.loads(line).get("metadata", {})
            repo = f"{meta.get('repo_owner')}/{meta.get('repo_name')}"
            if exclude_repo is not None and repo == exclude_repo:
                continue
            repo_counts[repo] += 1
    total = sum(repo_counts.values())
    top_repo, top_count = repo_counts.most_common(1)[0] if repo_counts else (None, 0)
    hhi = sum((count / total) ** 2 for count in repo_counts.values()) if total else 0.0
    top5 = sum(count for _, count in repo_counts.most_common(5))
    top10 = sum(count for _, count in repo_counts.most_common(10))
    return {
        "total": total,
        "unique_repos": len(repo_counts),
        "top_repo": top_repo,
        "top_count": top_count,
        "top_share": top_count / total if total else 0.0,
        "top5_share": top5 / total if total else 0.0,
        "top10_share": top10 / total if total else 0.0,
        "hhi": hhi,
        "repo_counts": repo_counts,
    }

def print_repo_summary(label, summary):
    print(label)
    print("=" * len(label))
    print(f"entries        : {summary['total']:,}")
    print(f"unique repos   : {summary['unique_repos']:,}")
    print(f"top repo       : {summary['top_repo']} ({summary['top_count']:,}, {100 * summary['top_share']:.1f}%)")
    print(f"top-5 share    : {100 * summary['top5_share']:.1f}%")
    print(f"top-10 share   : {100 * summary['top10_share']:.1f}%")
    print(f"HHI            : {summary['hhi']:.4f}")
    print("top repositories:")
    for repo, count in summary['repo_counts'].most_common(15):
        print(f"  {repo}: {count:,}")
    print()

strict_repo_summary = summarize_repo_concentration(cleaned_strict_core)
extended_repo_summary = summarize_repo_concentration(cleaned_extended_core)
extended_without_qdp_fsl_summary = summarize_repo_concentration(
    cleaned_extended_core,
    exclude_repo="backordinary/QDP-FSL",
)

print_repo_summary("Cleaned strict benchmark concentration", strict_repo_summary)
print_repo_summary("Cleaned extended benchmark concentration", extended_repo_summary)
print_repo_summary("Cleaned extended benchmark concentration without QDP-FSL", extended_without_qdp_fsl_summary)


## Post-Processing L — Corrected Validation and Benchmark-Suitability Summary

This final summary cell performs two tasks. First, it documents the correction from the earlier inflated broad `validated` headline to the current `materialized_circuit`-aware counts. Second, it reads the benchmark-annotated exports and prints the full benchmark-suitability score distribution as `n/7`, so that the numerical meaning of the strict core, extended core, validated reserve, and unvalidated Tier 2 groups is explicit.

The cell therefore serves as the publication-facing numerical summary for the rebuilt corpus, including the mutation-cleaned benchmark exports.

**Principal variables and helpers defined in the code cell**

- `status_counts`: corrected validation-status distribution for the final enriched pool
- `score_counts` and `tier_counts`: distributions over `benchmark_checks_passed / benchmark_checks_total` and `benchmark_suitability_tier`
- `tier_score_counts`: per-tier score-profile summary used to explain the `n/7` metric numerically
- `check_order` and `check_descriptions`: current seven benchmark-suitability checks and their textual interpretation
- `materialized`, `validated_nonzero_gate`, `validated_zero_gate`: key corrected validation tallies
- `old_validated_claim`, `new_validated`, and `shift`: quantities used to make the pre-fix versus post-fix comparison explicit
- `iter_jsonl()`, `line_count()`, and `format_score_profile()`: helpers used to stream JSONL files, verify export sizes, and summarize score distributions directly from disk


In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())

enriched_phase3 = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
strict_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_enriched.jsonl"
strict_tier2 = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_enriched.jsonl"
extended_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_enriched.jsonl"
strict_core_cleaned = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_cleaned_enriched.jsonl"
extended_core_cleaned = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_cleaned_enriched.jsonl"

check_order = [
    "validated_execution",
    "high_extraction_confidence",
    "no_demo_scaffolding",
    "no_cleanup_candidate",
    "minimum_code_lines",
    "minimum_gate_count",
    "trusted_retrieval_strategy",
]

check_descriptions = {
    "validated_execution": 'validation_status == "validated"',
    "high_extraction_confidence": 'extraction_confidence == "high"',
    "no_demo_scaffolding": "contains_demo_scaffolding != True",
    "no_cleanup_candidate": "cleanup_candidate != True",
    "minimum_code_lines": "code_lines >= 5",
    "minimum_gate_count": "gate_count >= 2",
    "trusted_retrieval_strategy": 'retrieval_strategy != "empirical_promoted_repo"',
}

status_counts = Counter()
score_counts = Counter()
tier_counts = Counter()
tier_score_counts = defaultdict(Counter)
materialized = 0
validated_nonzero_gate = 0
validated_zero_gate = 0
total = 0

def iter_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def line_count(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

def format_score_profile(counter, checks_total):
    parts = []
    for passed in range(checks_total, -1, -1):
        count = counter.get(passed, 0)
        if count:
            parts.append(f"{passed}/{checks_total}={count:,}")
    return ", ".join(parts) if parts else "none"

for obj in iter_jsonl(enriched_phase3):
    total += 1
    meta = obj.get("metadata", {})
    status = meta.get("validation_status", "<missing>")
    status_counts[status] += 1

    if meta.get("materialized_circuit") is True:
        materialized += 1

    if status == "validated":
        if (meta.get("gate_count") or 0) > 0:
            validated_nonzero_gate += 1
        else:
            validated_zero_gate += 1

for annotated_path in (strict_core, strict_tier2):
    for obj in iter_jsonl(annotated_path):
        meta = obj.get("metadata", {})
        passed = meta.get("benchmark_checks_passed")
        checks_total = meta.get("benchmark_checks_total")
        tier = meta.get("benchmark_suitability_tier", "<missing>")

        tier_counts[tier] += 1
        if passed is not None and checks_total is not None:
            score_counts[(passed, checks_total)] += 1
            tier_score_counts[tier][passed] += 1

old_validated_claim = 43021  # pre-fix figure from old Phase 3 report
new_validated = status_counts["validated"]
shift = old_validated_claim - new_validated
strict_core_count = line_count(strict_core)
extended_core_count = line_count(extended_core)
strict_core_cleaned_count = line_count(strict_core_cleaned)
extended_core_cleaned_count = line_count(extended_core_cleaned)
strict_core_mutation_delta = strict_core_count - strict_core_cleaned_count
extended_core_mutation_delta = extended_core_count - extended_core_cleaned_count
checks_total = max((total_checks for _, total_checks in score_counts), default=0)

print("Phase 3 corrected summary")
print("=" * 40)
print(f"raw total entries                 : {total:,}")
print(f"old reported validated            : {old_validated_claim:,}")
print(f"corrected validated               : {new_validated:,}")
print(f"reclassified out of validated     : {shift:,}")
print()
print("Corrected validation status counts:")
for k, v in status_counts.most_common():
    print(f"  {k}: {v:,}")
print()
print(f"materialized_circuit=True         : {materialized:,}")
print(f"validated and gate_count > 0      : {validated_nonzero_gate:,}")
print(f"validated and gate_count == 0     : {validated_zero_gate:,}")
print()
print(f"strict core export                : {strict_core_count:,}")
print(f"extended core export              : {extended_core_count:,}")
print(f"cleaned strict core export        : {strict_core_cleaned_count:,}")
print(f"cleaned extended core export      : {extended_core_cleaned_count:,}")
print(f"strict-core mutation exclusions   : {strict_core_mutation_delta:,}")
print(f"extended-core mutation exclusions : {extended_core_mutation_delta:,}")
print()
print(f"Benchmark suitability checks (current profile = {checks_total}):")
for idx, check_id in enumerate(check_order, 1):
    print(f"  {idx}. {check_id}: {check_descriptions[check_id]}")
print()
print(f"Benchmark suitability score distribution (n/{checks_total}):")
for passed in range(checks_total, -1, -1):
    print(f"  {passed}/{checks_total}: {score_counts.get((passed, checks_total), 0):,}")
print()
print("Tier interpretation:")
print(
    f"  strict_core_candidate         : {tier_counts['strict_core_candidate']:,} "
    f"(score profile: {format_score_profile(tier_score_counts['strict_core_candidate'], checks_total)})"
)
print(
    f"  extended_core_candidate       : {tier_counts['extended_core_candidate']:,} "
    f"(score profile: {format_score_profile(tier_score_counts['extended_core_candidate'], checks_total)}; "
    f"sole failed check = trusted_retrieval_strategy)"
)
print(
    f"  validated_broad_candidate     : {tier_counts['validated_broad_candidate']:,} "
    f"(score profile: {format_score_profile(tier_score_counts['validated_broad_candidate'], checks_total)})"
)
print(
    f"  tier2_unvalidated             : {tier_counts['tier2_unvalidated']:,} "
    f"(score profile: {format_score_profile(tier_score_counts['tier2_unvalidated'], checks_total)})"
)
print()
print("Practical reading:")
print(f"  strict core export = {strict_core_count:,} entries, which correspond to the 7/7 highest-trust subset")
print(
    f"  extended core export = {extended_core_count:,} entries, comprising the strict core plus "
    f"{tier_counts['extended_core_candidate']:,} entries that score 6/7 because they fail only the provenance-trust check"
)
print(
    f"  cleaned strict core export = {strict_core_cleaned_count:,} entries after removing mutation-suite paths "
    f"({strict_core_mutation_delta:,} entries removed from the original strict core)"
)
print(
    f"  cleaned extended core export = {extended_core_cleaned_count:,} entries after removing mutation-suite paths "
    f"({extended_core_mutation_delta:,} entries removed from the original extended core)"
)


## Master Processable Corpus Construction

This section establishes the working corpus that proceeds through seed generation, paraphrasing, and later semantic analyses before any benchmark, balance, or public-release filters are applied. The purpose is to keep the notebook transparent: reviewers can reproduce the full processable corpus first and then reconstruct every downstream split from explicit metadata flags rather than from an early hidden branch.

The master processable corpus is derived from the full enriched Phase 3 pool and keeps only entries that are operationally processable for later instruction generation. By default this means `validation_status == "validated"`, `materialized_circuit == True`, and `gate_count > 0`. Each retained record also carries benchmark-suitability annotations and mutation-suite flags so later benchmark and release filters can be reconstructed from the same master corpus.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
PYTHON_EXE = Path(sys.executable).resolve()

phase3_broad = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
phase3_master = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_master_processable_enriched.jsonl"
phase3_master_report = ROOT / "PQID/data/processed/master_processable_report_phase3.md"

print("python ready:", PYTHON_EXE.exists())
print("broad enriched file present:", phase3_broad.exists())
print("master corpus file present:", phase3_master.exists())
print("master report present:", phase3_master_report.exists())


### Master A — Build the Master Processable Corpus

This cell materializes the master processable corpus from the full corrected Phase 3 enriched pool. It does not create a benchmark subset. Instead, it produces the full validated, materialized, nonzero-gate working corpus that will later feed seed generation, paraphrasing, and semantic analysis.

When an older master file already exists, the rebuild script now carries forward master-only metadata fields such as `circuit_family` and `semantic_intent` where possible. This makes rebuilds more stable after later-stage enrichment. Even so, a deliberate rerun of `Pre-Seed B` and `Pre-Seed C` remains the safest way to refresh repository-level and semantic metadata after a master rebuild.

**Principal variables and artifacts used in the code cell**

- `phase3_broad`: full corrected enriched Phase 3 pool
- `phase3_master`: master processable corpus written for all downstream generation stages
- `phase3_master_report`: markdown summary of the processability rule and the retained corpus composition


In [ ]:
result = subprocess.run(
    [
        str(PYTHON_EXE),
        str(ROOT / "PQID/scripts/build_master_processable_corpus.py"),
        "--input-file",
        str(phase3_broad),
        "--output-file",
        str(phase3_master),
        "--report-file",
        str(phase3_master_report),
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("master-processable build completed with return code:", result.returncode)

### Master B — Structural Verification Before Pre-Seed Metadata Refresh

This verification cell is an intermediate structural checkpoint, not the final metadata-freeze checkpoint. Its purpose is to confirm that the master processable corpus was built correctly and that the working corpus already has the expected readiness, mutation, and size profiles before repository-level metadata is refreshed.

Because `Pre-Seed A` to `Pre-Seed C` have not necessarily been rerun yet when this cell is used, repository-level fields such as `repo_topics`, `repo_license`, `license_category`, `circuit_family`, and `semantic_intent` may still be stale here. The final post-refresh verification is therefore deferred to `Master C`, which appears after `Pre-Seed D`.

Both readiness views are reported here. The historical `n/7` score remains visible for continuity with the Phase 3 analysis, while the cleanliness-aware `n/8` score shows how the mutation-suite dimension changes the late-stage packaging view without changing the underlying working corpus.

**Principal variables used in the code cell**

- `master_status_counts`: validation-status distribution inside the retained master corpus
- `master_tier_counts` and `master_tier_counts_v2`: benchmark-suitability tiers retained inside the master corpus under the `n/7` and `n/8` views
- `master_score_counts` and `master_score_counts_v2`: score distributions under the original and cleanliness-aware readiness profiles
- `master_mutation_counts`: mutation-suite flag distribution carried forward for later filtering
- `master_license_counts`: currently visible license-category snapshot, which should be treated as provisional until `Pre-Seed B` has been rerun


In [ ]:
import json
from collections import Counter

master_status_counts = Counter()
master_tier_counts = Counter()
master_tier_counts_v2 = Counter()
master_score_counts = Counter()
master_score_counts_v2 = Counter()
master_mutation_counts = Counter()
master_license_counts = Counter()
total = 0

with open(phase3_master, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total += 1
        meta = json.loads(line).get("metadata", {})
        score_v1 = meta.get("benchmark_checks_passed")
        score_v2 = meta.get("benchmark_checks_passed_v2")
        if score_v2 is None and score_v1 is not None:
            score_v2 = score_v1 + (1 if meta.get("mutation_suite_candidate") is not True else 0)

        master_status_counts[meta.get("validation_status", "<missing>")] += 1
        master_tier_counts[meta.get("benchmark_suitability_tier", "<missing>")] += 1
        master_tier_counts_v2[meta.get("benchmark_suitability_tier_v2", "<missing>")] += 1
        if score_v1 is not None:
            master_score_counts[score_v1] += 1
        if score_v2 is not None:
            master_score_counts_v2[score_v2] += 1
        master_mutation_counts[meta.get("mutation_suite_candidate")] += 1
        master_license_counts[meta.get("license_category") or "<missing>"] += 1

print("master processable corpus structural snapshot (pre-seed refresh)")
print("=" * 40)
print(f"entries kept                    : {total:,}")
print()
print("validation status counts:")
for key, value in master_status_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("benchmark suitability tiers inside master corpus (n/7):")
for key, value in master_tier_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("benchmark suitability score distribution (n/7):")
for passed in sorted(master_score_counts, reverse=True):
    print(f"  {passed}/7: {master_score_counts[passed]:,}")
print()
print("cleanliness-aware tiers inside master corpus (n/8):")
for key, value in master_tier_counts_v2.most_common():
    print(f"  {key}: {value:,}")
print()
print("cleanliness-aware score distribution (n/8):")
for passed in sorted(master_score_counts_v2, reverse=True):
    print(f"  {passed}/8: {master_score_counts_v2[passed]:,}")
print()
print("mutation-suite flag distribution:")
for key, value in master_mutation_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("license distribution currently visible in master corpus (provisional before Pre-Seed B rerun):")
for key, value in master_license_counts.most_common():
    print(f"  {key}: {value:,}")


## Pre-Seed Metadata Population

This section enriches the full working corpus rather than a prefiltered benchmark subset. Repository topics and license metadata are patched onto both the full enriched Phase 3 pool and the master processable corpus so that the acquisition artifact and the downstream working corpus stay synchronized. Circuit family and semantic intent are then added to the master processable corpus, because that is the full corpus that will later flow into seed generation, paraphrasing, and semantic-consistency analysis.

Benchmark, balanced, and public-release subsets are intentionally deferred to the end of the pipeline. They are reconstructed later from metadata and audit flags attached to the master corpus rather than used as early-generation inputs.


In [ ]:
from pathlib import Path
import subprocess
import sys
import os

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
PYTHON_EXE = Path(sys.executable).resolve()

phase3_broad = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_enriched.jsonl"
phase3_master = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_master_processable_enriched.jsonl"
phase3_master_report = ROOT / "PQID/data/processed/master_processable_report_phase3.md"

preseed_repo_files = [
    phase3_broad,
    phase3_master,
]

preseed_semantic_files = [
    phase3_master,
]

print("python ready:", PYTHON_EXE.exists())
print("broad file present:", phase3_broad.exists())
print("master file present:", phase3_master.exists())
print("master report present:", phase3_master_report.exists())
print("github token env available:", bool(os.environ.get("GITHUB_TOKEN", "").strip()))
print("openai key env available:", bool(os.environ.get("OPENAI_API_KEY", "").strip()))
print("external secret dir present:", USER_SECRET_DIR.exists())
print("github token file available:", any(p.is_file() for p in GITHUB_TOKEN_CANDIDATES))
print("openai key file available:", any(p.is_file() for p in OPENAI_API_KEY_CANDIDATES))


### Pre-Seed Setup A — Secret File and Override Resolution

This cell checks the three supported secret-discovery mechanisms used by the pre-seed enrichment scripts: repository-local fallback files, an external user-level secret directory at `~/.pqid_secrets/`, and explicit override paths provided through `GITHUB_TOKEN_FILE` and `OPENAI_API_KEY_FILE`. The override variables refer to environment-variable names, not to literal filenames.


In [ ]:
import os
from pathlib import Path

def resolve_override(env_name):
    raw = os.environ.get(env_name, "").strip()
    if not raw:
        return None
    candidate = Path(raw).expanduser()
    return candidate if candidate.is_file() else None

github_override = resolve_override("GITHUB_TOKEN_FILE")
openai_override = resolve_override("OPENAI_API_KEY_FILE")

print("github token fallback files available:", any(p.is_file() for p in GITHUB_TOKEN_CANDIDATES))
print("openai key fallback files available:", any(p.is_file() for p in OPENAI_API_KEY_CANDIDATES))
print("github token override configured:", github_override is not None)
print("openai key override configured:", openai_override is not None)


### Pre-Seed A — Repository Topics and Ownership Context

This cell populates two repository-context metadata fields:

- `repo_topics`
- `is_org_repo`

These fields are derived from the GitHub repository metadata associated with each circuit's source file. They are useful for provenance analysis and later dataset reporting, but they do not depend on instruction text and should therefore be added before seed generation.

The enrichment is run over both the full corrected Phase 3 pool and the master processable corpus so that the acquisition artifact and the downstream working corpus remain synchronized.


In [ ]:
result = subprocess.run(
    [
        str(PYTHON_EXE),
        str(ROOT / "PQID/scripts/enrich_repo_topics.py"),
        "--input-files",
        *[str(p) for p in preseed_repo_files],
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("repo-topics enrichment completed with return code:", result.returncode)

### Pre-Seed B — Repository License Metadata

This cell populates the repository-license fields:

- `repo_license`
- `license_category`

These fields support release governance, filtering, and downstream reporting. Because they are repository-level provenance attributes rather than instruction-level attributes, they should be attached before seed generation.

The enrichment is run on both the full corrected Phase 3 pool and the master processable corpus so that later release filtering can be reproduced from the same fully processed records.

If an upstream repository changes its license during the audit period, the local license cache may contain a stale record. The execution cell therefore supports a small `license_refresh_owner_repos` list for forcing a targeted refresh of known changed repositories before patching the JSONL files.


In [ ]:
license_refresh_owner_repos = [
    "Ahmik-Virani/Differentiating-Quantum-Bug-From-Noise-Statistical-Approach",
]

cmd = [
    str(PYTHON_EXE),
    str(ROOT / "PQID/scripts/enrich_repo_license.py"),
    "--input-files",
    *[str(p) for p in preseed_repo_files],
]

if license_refresh_owner_repos:
    cmd.extend(["--refresh-owner-repos", *license_refresh_owner_repos])

result = subprocess.run(
    cmd,
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("repo-license enrichment completed with return code:", result.returncode)

### Pre-Seed C — Circuit Family and Semantic Intent

This cell classifies the full master processable corpus into structural families and semantic intent categories by populating:

- `circuit_family`
- `semantic_intent`

Unlike repository topics or license metadata, this enrichment uses an LLM and therefore has nontrivial cost. For that reason, the default workflow applies it to the master processable corpus only, which is the actual working corpus for the later seed/paraphrase/semantic pipeline.

The script expects the current notebook environment to provide both the `openai` package and a valid OpenAI credential through `OPENAI_API_KEY`, `OPENAI_API_KEY_FILE`, or an untracked local key file. The execution cell therefore performs a short preflight check before launching the enrichment subprocess.

Important workflow note: if `Master A` has been rerun after an earlier semantic pass, rerunning this cell is still cheap because the classifications are already stored in `circuit_family_cache.jsonl`. In that case the script mainly repatches the rebuilt master file from cache rather than reclassifying circuits from scratch.


In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

assert importlib.util.find_spec("openai") is not None, (
    "The current notebook interpreter is missing the 'openai' package. Install it into this environment before running this cell."
)

env = os.environ.copy()

def override_points_to_file(env_name):
    raw = env.get(env_name, "").strip()
    if not raw:
        return False
    return Path(raw).expanduser().is_file()


def named_openai_key_available():
    for filename in OPENAI_NAMED_SECRET_FILENAMES:
        for root in NAMED_SECRET_SEARCH_ROOTS:
            if not root.exists():
                continue
            try:
                for candidate in root.rglob(filename):
                    if candidate.is_file():
                        return True
            except (OSError, PermissionError):
                continue
    return False

has_openai_credential = (
    bool(env.get("OPENAI_API_KEY", "").strip())
    or override_points_to_file("OPENAI_API_KEY_FILE")
    or any(p.is_file() for p in OPENAI_API_KEY_CANDIDATES)
    or named_openai_key_available()
)

assert has_openai_credential, (
    "OpenAI key not configured. Set OPENAI_API_KEY, set OPENAI_API_KEY_FILE, or create ~/.pqid_secrets/openai_api_key.txt"
)

result = subprocess.run(
    [
        str(PYTHON_EXE),
        str(ROOT / "PQID/scripts/enrich_circuit_family.py"),
        "--input-files",
        *[str(p) for p in preseed_semantic_files],
    ],
    env=env,
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("circuit-family enrichment completed with return code:", result.returncode)

### Pre-Seed D — Verification of Metadata Population

This verification cell checks that the newly added pre-seed metadata fields are populated in the master processable corpus. It is a targeted field-completeness check and should be read together with `Master C`, which provides the final post-refresh structural and metadata-freeze snapshot.


In [ ]:
import json
from collections import Counter

field_counts = Counter()
total = 0

with open(phase3_master, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total += 1
        meta = json.loads(line).get("metadata", {})
        for field in [
            "repo_topics",
            "is_org_repo",
            "repo_license",
            "license_category",
            "circuit_family",
            "semantic_intent",
        ]:
            if meta.get(field) not in (None, "", []):
                field_counts[field] += 1

print("entries checked:", total)
for field in [
    "repo_topics",
    "is_org_repo",
    "repo_license",
    "license_category",
    "circuit_family",
    "semantic_intent",
]:
    print(f"{field:20s}: {field_counts[field]:,}")


### Master C — Final Verification After Pre-Seed Metadata Refresh

This is the final master-corpus checkpoint after `Pre-Seed A`, `Pre-Seed B`, and `Pre-Seed C` have been rerun. Unlike `Master B`, this cell is intended to be used as the metadata-freeze view before seed generation begins.

The expectation is that structural counts remain stable relative to `Master B`, while repository-level metadata such as license categories, circuit family labels, and semantic intent labels are no longer stale. This cell therefore combines the structural corpus summary with the key pre-seed metadata completeness counts in one place.

**Principal variables used in the code cell**

- `master_status_counts`, `master_tier_counts`, and `master_tier_counts_v2`: final structural distributions inside the master corpus
- `master_score_counts` and `master_score_counts_v2`: final `n/7` and `n/8` readiness profiles used for downstream filtering interpretation
- `master_mutation_counts`: mutation-suite split retained for late-stage corpus-role decisions
- `master_license_counts`: refreshed license-category snapshot after rerunning `Pre-Seed B`
- `field_counts`: completeness counts for the key pre-seed metadata fields that should be frozen before seed generation


In [ ]:
from pathlib import Path
import json
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
phase3_master = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_master_processable_enriched.jsonl"

master_status_counts = Counter()
master_tier_counts = Counter()
master_tier_counts_v2 = Counter()
master_score_counts = Counter()
master_score_counts_v2 = Counter()
master_mutation_counts = Counter()
master_license_counts = Counter()
field_counts = Counter()
total = 0

fields_to_check = [
    "repo_topics",
    "is_org_repo",
    "repo_license",
    "license_category",
    "circuit_family",
    "semantic_intent",
]

with open(phase3_master, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total += 1
        meta = json.loads(line).get("metadata", {})
        score_v1 = meta.get("benchmark_checks_passed")
        score_v2 = meta.get("benchmark_checks_passed_v2")
        if score_v2 is None and score_v1 is not None:
            score_v2 = score_v1 + (1 if meta.get("mutation_suite_candidate") is not True else 0)

        master_status_counts[meta.get("validation_status", "<missing>")] += 1
        master_tier_counts[meta.get("benchmark_suitability_tier", "<missing>")] += 1
        master_tier_counts_v2[meta.get("benchmark_suitability_tier_v2", "<missing>")] += 1
        if score_v1 is not None:
            master_score_counts[score_v1] += 1
        if score_v2 is not None:
            master_score_counts_v2[score_v2] += 1
        master_mutation_counts[meta.get("mutation_suite_candidate")] += 1
        master_license_counts[meta.get("license_category") or "<missing>"] += 1

        for field in fields_to_check:
            if meta.get(field) not in (None, "", []):
                field_counts[field] += 1

print("final master processable corpus summary after pre-seed refresh")
print("=" * 40)
print(f"entries kept                    : {total:,}")
print()
print("validation status counts:")
for key, value in master_status_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("benchmark suitability tiers inside master corpus (n/7):")
for key, value in master_tier_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("benchmark suitability score distribution (n/7):")
for passed in sorted(master_score_counts, reverse=True):
    print(f"  {passed}/7: {master_score_counts[passed]:,}")
print()
print("cleanliness-aware tiers inside master corpus (n/8):")
for key, value in master_tier_counts_v2.most_common():
    print(f"  {key}: {value:,}")
print()
print("cleanliness-aware score distribution (n/8):")
for passed in sorted(master_score_counts_v2, reverse=True):
    print(f"  {passed}/8: {master_score_counts_v2[passed]:,}")
print()
print("mutation-suite flag distribution:")
for key, value in master_mutation_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("license distribution visible in master corpus:")
for key, value in master_license_counts.most_common():
    print(f"  {key}: {value:,}")
print()
print("key pre-seed metadata completeness:")
for field in fields_to_check:
    print(f"  {field:20s}: {field_counts[field]:,}")


# Statistical Analysis of the Benchmark-Suitability Distribution

The benchmark-suitability analysis now has two complementary layers. The original `n/7` score remains the historically continuous readiness metric used in the Phase 3 correction analysis, so the main statistical battery below continues to analyze that seven-check profile directly. Those cells answer the original question: can the observed readiness distribution be explained as a simple accumulation of heterogeneous pass/fail criteria, or does it reveal structured failure modes in the dataset?

The analysis is carried out on the benchmark-annotated strict-core and strict-Tier-2 exports together, because those two files jointly cover the full corrected Phase 3 corpus and preserve the per-entry suitability annotations.

Three complementary questions are addressed for the original `n/7` profile:

1. What are the empirical pass rates of the seven individual checks, and how do they combine into the observed score distribution?
2. Does the observed score distribution differ from the Poisson-binomial null implied by independent heterogeneous checks?
3. Is the pronounced `6/7` shoulder driven primarily by a single bottleneck criterion, in particular the provenance-trust check?

A separate companion subsection later in this notebook extends the analysis to the cleanliness-aware `n/8` view on the master corpus. That subsection does not replace the original `n/7` tests; instead, it shows how the added mutation-suite criterion changes the packaging and release interpretation once the historical benchmark-readiness structure is already understood.


## Statistical Analysis A — Empirical Check Structure

This cell reconstructs the seven binary benchmark-suitability checks for every entry in the corrected Phase 3 corpus. It then prints the empirical pass rate of each check, together with the observed `n/7` score distribution.

The resulting check matrix is reused by the later cells for formal statistical testing.

**Principal variables introduced in the code cell**

- `analysis_files`: the strict-core and strict-Tier-2 files whose union covers the corrected benchmark-annotated corpus
- `check_order` and `check_descriptions`: the seven suitability checks in evaluation order and their textual interpretation
- `X`: binary check matrix with one row per circuit and one column per check
- `scores`: vector of readiness scores obtained by summing the seven binary checks
- `pass_rates`: empirical marginal pass probabilities used later to fit the Poisson-binomial null


In [ ]:
from pathlib import Path
import json
from collections import Counter

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())

strict_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_enriched.jsonl"
strict_tier2 = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_enriched.jsonl"
analysis_files = [strict_core, strict_tier2]

check_order = [
    "validated_execution",
    "high_extraction_confidence",
    "no_demo_scaffolding",
    "no_cleanup_candidate",
    "minimum_code_lines",
    "minimum_gate_count",
    "trusted_retrieval_strategy",
]

check_descriptions = {
    "validated_execution": 'validation_status == "validated"',
    "high_extraction_confidence": 'extraction_confidence == "high"',
    "no_demo_scaffolding": "contains_demo_scaffolding != True",
    "no_cleanup_candidate": "cleanup_candidate != True",
    "minimum_code_lines": "code_lines >= 5",
    "minimum_gate_count": "gate_count >= 2",
    "trusted_retrieval_strategy": 'retrieval_strategy != "empirical_promoted_repo"',
}

def iter_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

rows = []
tier_counts = Counter()

for path in analysis_files:
    for obj in iter_jsonl(path):
        meta = obj.get("metadata", {})
        passed_checks = set(meta.get("benchmark_passed_checks") or [])
        row = [1 if cid in passed_checks else 0 for cid in check_order]
        rows.append(row)
        tier_counts[meta.get("benchmark_suitability_tier", "<missing>")] += 1

X = np.asarray(rows, dtype=np.int8)
scores = X.sum(axis=1)
pass_rates = X.mean(axis=0)
observed_score_counts = np.bincount(scores, minlength=len(check_order) + 1)

print("Empirical pass rates by benchmark check")
print("=" * 48)
for idx, (cid, p) in enumerate(zip(check_order, pass_rates), 1):
    print(f"{idx}. {cid:28s} {p:7.4f}   {check_descriptions[cid]}")

print()
print("Observed benchmark-suitability distribution")
print("=" * 48)
for passed in range(len(check_order), -1, -1):
    print(f"  {passed}/{len(check_order)}: {observed_score_counts[passed]:,}")

print()
print("Tier counts in the annotated corpus")
print("=" * 48)
for tier, count in tier_counts.most_common():
    print(f"  {tier:28s} {count:,}")


## Statistical Analysis B — Poisson-Binomial Null and Overdispersion

Under a simple null model, the benchmark-readiness score is the sum of seven independent Bernoulli checks with unequal pass probabilities equal to their empirical marginals. The resulting score distribution is therefore Poisson-binomial rather than binomial.

This cell fits that null model, computes the expected `0/7` to `7/7` counts, and compares them with the observed counts. Two summary diagnostics are reported:

- a Pearson chi-square statistic with a Monte Carlo p-value based on multinomial sampling from the fitted null
- an overdispersion ratio comparing the observed variance of the readiness score with the variance implied by the independent-check null

A low p-value and an overdispersion ratio above 1.0 indicate that the distribution is not explained by independent heterogeneous checks alone, and therefore reflects structured dependence or latent subpopulations in the dataset.

**Principal variables introduced in the code cell**

- `pb_pmf`: exact Poisson-binomial probability mass function fitted from the empirical pass rates
- `expected_score_counts`: expected counts under the independent-check null
- `chi2_stat` and `bootstrap_p_value`: goodness-of-fit statistic and Monte Carlo p-value
- `mean_obs`, `var_obs`, `mean_null`, `var_null`, and `overdispersion_ratio`: summary moments used to evaluate score concentration relative to the null


In [ ]:
def poisson_binomial_pmf(probs):
    pmf = np.array([1.0], dtype=float)
    for p in probs:
        next_pmf = np.zeros(len(pmf) + 1, dtype=float)
        next_pmf[:-1] += pmf * (1.0 - p)
        next_pmf[1:] += pmf * p
        pmf = next_pmf
    return pmf

pb_pmf = poisson_binomial_pmf(pass_rates)
expected_score_counts = len(scores) * pb_pmf

safe_expected = np.where(expected_score_counts > 0, expected_score_counts, 1.0)
chi2_stat = float(np.sum((observed_score_counts - expected_score_counts) ** 2 / safe_expected))

mean_obs = float(scores.mean())
var_obs = float(scores.var(ddof=0))
mean_null = float(pass_rates.sum())
var_null = float(np.sum(pass_rates * (1.0 - pass_rates)))
overdispersion_ratio = var_obs / var_null if var_null > 0 else float("nan")

rng = np.random.default_rng(42)
n_boot = 5000
boot_counts = rng.multinomial(len(scores), pb_pmf, size=n_boot)
boot_chi2 = ((boot_counts - expected_score_counts) ** 2 / safe_expected).sum(axis=1)
bootstrap_p_value = float((1 + np.sum(boot_chi2 >= chi2_stat)) / (n_boot + 1))

print("Poisson-binomial null fit")
print("=" * 48)
print(f"observed mean score              : {mean_obs:.4f}")
print(f"null mean score                  : {mean_null:.4f}")
print(f"observed score variance          : {var_obs:.4f}")
print(f"null score variance              : {var_null:.4f}")
print(f"overdispersion ratio             : {overdispersion_ratio:.4f}")
print(f"Pearson chi-square statistic     : {chi2_stat:.4f}")
print(f"Monte Carlo goodness-of-fit p    : {bootstrap_p_value:.6f}")

print()
print("Observed versus expected score counts under the null")
print("=" * 48)
for passed in range(len(check_order), -1, -1):
    observed = int(observed_score_counts[passed])
    expected = expected_score_counts[passed]
    diff = observed - expected
    print(f"  {passed}/{len(check_order)}: observed={observed:>6,}  expected={expected:>10.1f}  diff={diff:>10.1f}")


## Statistical Analysis C — Single-Failure Bottlenecks and the `6/7` Shoulder

The preceding goodness-of-fit test can show that the readiness distribution is not explained by independent check failures alone. This final cell addresses the substantive hypothesis behind the `6/7` shoulder: namely, that a large near-core subset fails only the provenance-trust criterion while passing all six substantive quality checks.

To test that hypothesis, the cell counts all circuits that fail exactly one check and identifies which check it is. It also computes pairwise phi coefficients between the provenance-trust check and the remaining six checks. If the `6/7` population is dominated by `trusted_retrieval_strategy`, then the shoulder is best interpreted as a provenance bottleneck rather than a general quality deficit.

**Principal variables introduced in the code cell**

- `single_failure_counts`: counts of circuits that fail exactly one of the seven checks
- `six_of_seven_total` and `trust_only_share`: size of the `6/7` population and the share explained by failure of the provenance-trust check alone
- `phi_against_trust`: pairwise phi coefficients comparing the provenance-trust check against every other criterion


In [ ]:
single_failure_counts = Counter()
for row in X:
    if row.sum() == len(check_order) - 1:
        failed_idx = int(np.where(row == 0)[0][0])
        single_failure_counts[check_order[failed_idx]] += 1

def phi_coefficient(a, b):
    a = np.asarray(a, dtype=int)
    b = np.asarray(b, dtype=int)
    n11 = int(np.sum((a == 1) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    n01 = int(np.sum((a == 0) & (b == 1)))
    n00 = int(np.sum((a == 0) & (b == 0)))
    denom = (n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)
    if denom == 0:
        return float("nan")
    return (n11 * n00 - n10 * n01) / np.sqrt(denom)

trust_idx = check_order.index("trusted_retrieval_strategy")
six_of_seven_total = int(np.sum(scores == len(check_order) - 1))
trust_only_share = (
    single_failure_counts["trusted_retrieval_strategy"] / six_of_seven_total
    if six_of_seven_total > 0 else float("nan")
)

phi_against_trust = {}
for idx, cid in enumerate(check_order):
    if cid == "trusted_retrieval_strategy":
        continue
    phi_against_trust[cid] = phi_coefficient(X[:, trust_idx], X[:, idx])

print("Single-failure bottleneck analysis")
print("=" * 48)
print(f"total 6/7 entries                           : {six_of_seven_total:,}")
print(f"fail only trusted_retrieval_strategy        : {single_failure_counts['trusted_retrieval_strategy']:,}")
print(f"share of 6/7 explained by trust-only failure: {trust_only_share:.4f}")

print()
print("Exact single-failure counts by check")
print("=" * 48)
for cid in check_order:
    print(f"  {cid:28s} {single_failure_counts[cid]:,}")

print()
print("Phi coefficients against trusted_retrieval_strategy")
print("=" * 48)
for cid, phi in sorted(phi_against_trust.items(), key=lambda kv: abs(kv[1]), reverse=True):
    print(f"  {cid:28s} {phi: .4f}")


## Statistical Analysis D — Full Pairwise Phi-Correlation Matrix

This final cell extends the targeted provenance analysis to the full seven-check system. It computes the complete pairwise phi-correlation matrix for all benchmark-suitability checks and, when plotting libraries are available, renders the matrix as a heatmap.

The phi coefficient is the standard association measure for binary variables. Positive values indicate that two checks tend to pass or fail together more often than expected under independence, while negative values indicate opposing behavior.

In the present setting, the matrix is useful for distinguishing between two possibilities:

1. the benchmark-readiness score behaves as a single coherent quality dimension, in which most checks positively cluster; or
2. the provenance-trust criterion behaves differently from the intrinsic quality checks, supporting the interpretation that the `6/7` shoulder reflects a distinct provenance bottleneck.

In the corrected Phase 3 corpus, the expected interpretation is the second one. Strong positive associations among `validated_execution`, `high_extraction_confidence`, and `minimum_gate_count` indicate a coherent intrinsic-quality cluster, while `no_demo_scaffolding` and `no_cleanup_candidate` form a second cleanup-related cluster. By contrast, `trusted_retrieval_strategy` is negatively associated with every other check, which is consistent with the earlier single-failure analysis showing that most `6/7` entries fail only the provenance-trust criterion. The matrix therefore corroborates the claim that the extended core is separated from the strict core primarily by source-trust constraints rather than by a broad deterioration in circuit quality.

**Principal variables introduced in the code cell**

- `phi_matrix`: full symmetric `7 x 7` matrix of phi coefficients across all benchmark checks
- `phi_rows`: text-formatted matrix rows for notebook-readable inspection even without plotting support
- `plt`: optional plotting backend used to render the heatmap when available


In [ ]:
phi_matrix = np.zeros((len(check_order), len(check_order)), dtype=float)
for i in range(len(check_order)):
    for j in range(len(check_order)):
        phi_matrix[i, j] = phi_coefficient(X[:, i], X[:, j])

print("Full pairwise phi-correlation matrix")
print("=" * 48)
header = " " * 30 + " ".join(f"{i+1:>7d}" for i in range(len(check_order)))
print(header)
for i, cid in enumerate(check_order):
    row_text = " ".join(f"{phi_matrix[i, j]:7.3f}" for j in range(len(check_order)))
    print(f"{i+1}. {cid:25s} {row_text}")

print()
print("Check index legend")
print("=" * 48)
for idx, cid in enumerate(check_order, 1):
    print(f"  {idx}. {cid}")

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(phi_matrix, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    ax.set_xticks(range(len(check_order)))
    ax.set_yticks(range(len(check_order)))
    ax.set_xticklabels([f"{i+1}" for i in range(len(check_order))])
    ax.set_yticklabels([f"{i+1}" for i in range(len(check_order))])
    ax.set_title("Pairwise phi-correlation across benchmark checks")

    for i in range(len(check_order)):
        for j in range(len(check_order)):
            ax.text(j, i, f"{phi_matrix[i, j]:.2f}", ha="center", va="center", color="black", fontsize=8)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("phi coefficient")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print()
    print(f"Heatmap skipped: {type(exc).__name__}: {exc}")


## Statistical Analysis E — Bootstrap Confidence Intervals

The preceding cells establish the qualitative structure of the benchmark-suitability distribution. This final subsection adds uncertainty quantification for the two most important scalar summaries:

1. the overdispersion ratio, which measures how much broader the observed score distribution is than the independent heterogeneous-check null; and
2. the trust-only share of the `6/7` group, which measures how much of the near-core shoulder is explained by failure of `trusted_retrieval_strategy` alone.

The cell uses nonparametric bootstrap resampling over circuits. Each bootstrap sample preserves the seven-check structure of individual entries but resamples rows of the empirical check matrix with replacement. For each resample, the cell recomputes the two statistics and then reports percentile-based 95% confidence intervals.

These intervals do not replace the earlier goodness-of-fit test; rather, they quantify the stability of the two summary effects most central to the interpretation of the readiness profile.

**Principal variables introduced in the code cell**

- `n_boot`: number of bootstrap resamples
- `bootstrap_seed`: random seed used for reproducibility
- `overdispersion_boot`: bootstrap distribution of the overdispersion ratio
- `trust_share_boot`: bootstrap distribution of the trust-only share within the `6/7` group
- `overdispersion_ci` and `trust_share_ci`: percentile-based 95% confidence intervals for the two headline statistics


In [ ]:
def bootstrap_summary_statistics(X_sample):
    sample_scores = X_sample.sum(axis=1)
    sample_pass_rates = X_sample.mean(axis=0)
    sample_obs_var = float(np.var(sample_scores, ddof=0))
    sample_null_var = float(np.sum(sample_pass_rates * (1.0 - sample_pass_rates)))
    sample_overdispersion = (
        sample_obs_var / sample_null_var if sample_null_var > 0 else float("nan")
    )

    trust_col = check_order.index("trusted_retrieval_strategy")
    six_mask = sample_scores == len(check_order) - 1
    six_total = int(np.sum(six_mask))
    trust_only = int(
        np.sum(
            six_mask
            & (X_sample[:, trust_col] == 0)
            & (np.sum(X_sample[:, np.arange(len(check_order)) != trust_col], axis=1) == len(check_order) - 1)
        )
    )
    sample_trust_share = trust_only / six_total if six_total > 0 else float("nan")
    return sample_overdispersion, sample_trust_share

bootstrap_seed = 0
n_boot = 2000
rng = np.random.default_rng(bootstrap_seed)

point_overdispersion, point_trust_share = bootstrap_summary_statistics(X)

overdispersion_boot = []
trust_share_boot = []

for _ in range(n_boot):
    sample_idx = rng.integers(0, X.shape[0], size=X.shape[0])
    X_boot = X[sample_idx]
    boot_overdispersion, boot_trust_share = bootstrap_summary_statistics(X_boot)
    overdispersion_boot.append(boot_overdispersion)
    trust_share_boot.append(boot_trust_share)

overdispersion_boot = np.asarray(overdispersion_boot, dtype=float)
trust_share_boot = np.asarray(trust_share_boot, dtype=float)

overdispersion_boot = overdispersion_boot[np.isfinite(overdispersion_boot)]
trust_share_boot = trust_share_boot[np.isfinite(trust_share_boot)]

overdispersion_ci = np.percentile(overdispersion_boot, [2.5, 97.5])
trust_share_ci = np.percentile(trust_share_boot, [2.5, 97.5])

print("Bootstrap confidence intervals")
print("=" * 48)
print(f"bootstrap resamples                       : {n_boot:,}")
print(f"bootstrap seed                            : {bootstrap_seed}")
print()
print("Overdispersion ratio")
print("-" * 48)
print(f"point estimate                            : {point_overdispersion:.4f}")
print(f"95% percentile CI                         : [{overdispersion_ci[0]:.4f}, {overdispersion_ci[1]:.4f}]")
print()
print("Trust-only share within the 6/7 shoulder")
print("-" * 48)
print(f"point estimate                            : {point_trust_share:.4f}")
print(f"95% percentile CI                         : [{trust_share_ci[0]:.4f}, {trust_share_ci[1]:.4f}]")


## Statistical Analysis F — Cleanliness-Aware `n/8` Companion Diagnostics

The `n/8` score is not a replacement for the original readiness metric. Instead, it adds one explicit late-stage cleanliness criterion, `non_mutation_suite_path`, so that users can distinguish intrinsically strong benchmark candidates from mutation-suite or bug-stress entries without losing the earlier `n/7` interpretation.

Because of that design choice, the appropriate statistical question is slightly different from the earlier `n/7` battery. This cell therefore treats the `n/8` view as a companion analysis on the fully processable master corpus and asks four targeted questions:

1. How do entries transition from the original `n/7` tier labels into the cleanliness-aware `n/8` tier labels?
2. Does the `n/8` score distribution also depart from the independent heterogeneous-check null?
3. Is the `7/8` shoulder dominated by failure of the added mutation-suite criterion alone?
4. How strongly is `non_mutation_suite_path` associated with the original seven checks?

The resulting output should be read as a packaging-aware extension of the original benchmark-readiness analysis: the `n/7` battery explains the historical benchmark structure, while this `n/8` companion diagnostic shows how explicit contamination control restructures the usable corpus.

**Principal variables introduced in the code cell**

- `check_order_v2`: the eight-check cleanliness-aware readiness profile
- `X_v2` and `scores_v2`: the binary check matrix and score vector for the master corpus under `n/8`
- `transition_counts_v2`: transition table from the original `n/7` tiers to the cleanliness-aware `n/8` tiers
- `overdispersion_ratio_v2` and `bootstrap_p_value_v2`: null-fit diagnostics for the `n/8` distribution
- `single_failure_counts_v2` and `mutation_only_share_v2`: bottleneck summary for the `7/8` shoulder
- `phi_against_mutation`: pairwise phi coefficients comparing `non_mutation_suite_path` against the original seven checks
- `overdispersion_ci_v2` and `mutation_share_ci_v2`: percentile-based 95% bootstrap confidence intervals for the two headline `n/8` summary effects


In [ ]:
from pathlib import Path
import json
from collections import Counter

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
phase3_master = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_master_processable_enriched.jsonl"

check_order_v2 = [
    "validated_execution",
    "high_extraction_confidence",
    "no_demo_scaffolding",
    "no_cleanup_candidate",
    "minimum_code_lines",
    "minimum_gate_count",
    "trusted_retrieval_strategy",
    "non_mutation_suite_path",
]

check_descriptions_v2 = {
    "validated_execution": 'validation_status == "validated"',
    "high_extraction_confidence": 'extraction_confidence == "high"',
    "no_demo_scaffolding": "contains_demo_scaffolding != True",
    "no_cleanup_candidate": "cleanup_candidate != True",
    "minimum_code_lines": "code_lines >= 5",
    "minimum_gate_count": "gate_count >= 2",
    "trusted_retrieval_strategy": 'retrieval_strategy != "empirical_promoted_repo"',
    "non_mutation_suite_path": "mutation_suite_candidate != True",
}

def poisson_binomial_pmf_local(probs):
    pmf = np.array([1.0], dtype=float)
    for p in probs:
        next_pmf = np.zeros(len(pmf) + 1, dtype=float)
        next_pmf[:-1] += pmf * (1.0 - p)
        next_pmf[1:] += pmf * p
        pmf = next_pmf
    return pmf

def phi_coefficient_local(a, b):
    a = np.asarray(a, dtype=int)
    b = np.asarray(b, dtype=int)
    n11 = int(np.sum((a == 1) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    n01 = int(np.sum((a == 0) & (b == 1)))
    n00 = int(np.sum((a == 0) & (b == 0)))
    denom = (n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)
    if denom == 0:
        return float("nan")
    return (n11 * n00 - n10 * n01) / np.sqrt(denom)

rows_v2 = []
transition_counts_v2 = Counter()

def fmt_pct(num, den):
    return 100.0 * num / den if den else float("nan")

with open(phase3_master, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        meta = json.loads(line).get("metadata", {})
        passed_v2 = set(meta.get("benchmark_passed_checks_v2") or [])
        if not passed_v2:
            passed_v2 = set(meta.get("benchmark_passed_checks") or [])
            if meta.get("mutation_suite_candidate") is not True:
                passed_v2.add("non_mutation_suite_path")

        row_v2 = [1 if cid in passed_v2 else 0 for cid in check_order_v2]
        rows_v2.append(row_v2)

        transition_counts_v2[(
            meta.get("benchmark_suitability_tier", "<missing>"),
            meta.get("benchmark_suitability_tier_v2", "<missing>"),
        )] += 1

X_v2 = np.asarray(rows_v2, dtype=np.int8)
scores_v2 = X_v2.sum(axis=1)
pass_rates_v2 = X_v2.mean(axis=0)
observed_score_counts_v2 = np.bincount(scores_v2, minlength=len(check_order_v2) + 1)

pb_pmf_v2 = poisson_binomial_pmf_local(pass_rates_v2)
expected_score_counts_v2 = len(scores_v2) * pb_pmf_v2
safe_expected_v2 = np.where(expected_score_counts_v2 > 0, expected_score_counts_v2, 1.0)
chi2_stat_v2 = float(np.sum((observed_score_counts_v2 - expected_score_counts_v2) ** 2 / safe_expected_v2))

mean_obs_v2 = float(scores_v2.mean())
var_obs_v2 = float(scores_v2.var(ddof=0))
mean_null_v2 = float(pass_rates_v2.sum())
var_null_v2 = float(np.sum(pass_rates_v2 * (1.0 - pass_rates_v2)))
overdispersion_ratio_v2 = var_obs_v2 / var_null_v2 if var_null_v2 > 0 else float("nan")

rng = np.random.default_rng(42)
n_boot_null_v2 = 3000
boot_counts_v2 = rng.multinomial(len(scores_v2), pb_pmf_v2, size=n_boot_null_v2)
boot_chi2_v2 = ((boot_counts_v2 - expected_score_counts_v2) ** 2 / safe_expected_v2).sum(axis=1)
bootstrap_p_value_v2 = float((1 + np.sum(boot_chi2_v2 >= chi2_stat_v2)) / (n_boot_null_v2 + 1))

single_failure_counts_v2 = Counter()
for row in X_v2:
    if row.sum() == len(check_order_v2) - 1:
        failed_idx = int(np.where(row == 0)[0][0])
        single_failure_counts_v2[check_order_v2[failed_idx]] += 1

mutation_idx = check_order_v2.index("non_mutation_suite_path")
seven_of_eight_total = int(np.sum(scores_v2 == len(check_order_v2) - 1))
mutation_only_share_v2 = (
    single_failure_counts_v2["non_mutation_suite_path"] / seven_of_eight_total
    if seven_of_eight_total > 0 else float("nan")
)

phi_against_mutation = {}
for idx, cid in enumerate(check_order_v2):
    if cid == "non_mutation_suite_path":
        continue
    phi_against_mutation[cid] = phi_coefficient_local(X_v2[:, mutation_idx], X_v2[:, idx])

def bootstrap_summary_statistics_v2(X_sample):
    sample_scores = X_sample.sum(axis=1)
    sample_pass_rates = X_sample.mean(axis=0)
    sample_obs_var = float(np.var(sample_scores, ddof=0))
    sample_null_var = float(np.sum(sample_pass_rates * (1.0 - sample_pass_rates)))
    sample_overdispersion = sample_obs_var / sample_null_var if sample_null_var > 0 else float("nan")

    shoulder_mask = sample_scores == len(check_order_v2) - 1
    shoulder_total = int(np.sum(shoulder_mask))
    mutation_only = int(
        np.sum(
            shoulder_mask
            & (X_sample[:, mutation_idx] == 0)
            & (np.sum(X_sample[:, np.arange(len(check_order_v2)) != mutation_idx], axis=1) == len(check_order_v2) - 1)
        )
    )
    sample_mutation_share = mutation_only / shoulder_total if shoulder_total > 0 else float("nan")
    return sample_overdispersion, sample_mutation_share

bootstrap_seed_v2 = 0
n_boot_v2 = 2000
rng_v2 = np.random.default_rng(bootstrap_seed_v2)
point_overdispersion_v2, point_mutation_share_v2 = bootstrap_summary_statistics_v2(X_v2)

overdispersion_boot_v2 = []
mutation_share_boot_v2 = []
for _ in range(n_boot_v2):
    sample_idx = rng_v2.integers(0, X_v2.shape[0], size=X_v2.shape[0])
    X_boot = X_v2[sample_idx]
    boot_overdispersion, boot_mutation_share = bootstrap_summary_statistics_v2(X_boot)
    overdispersion_boot_v2.append(boot_overdispersion)
    mutation_share_boot_v2.append(boot_mutation_share)

overdispersion_boot_v2 = np.asarray(overdispersion_boot_v2, dtype=float)
mutation_share_boot_v2 = np.asarray(mutation_share_boot_v2, dtype=float)
overdispersion_boot_v2 = overdispersion_boot_v2[np.isfinite(overdispersion_boot_v2)]
mutation_share_boot_v2 = mutation_share_boot_v2[np.isfinite(mutation_share_boot_v2)]
overdispersion_ci_v2 = np.percentile(overdispersion_boot_v2, [2.5, 97.5])
mutation_share_ci_v2 = np.percentile(mutation_share_boot_v2, [2.5, 97.5])

print("Empirical pass rates by cleanliness-aware benchmark check")
print("=" * 64)
for idx, (cid, p) in enumerate(zip(check_order_v2, pass_rates_v2), 1):
    print(f"{idx}. {cid:28s} {p:7.4f}   {check_descriptions_v2[cid]}")

print()
print("Observed cleanliness-aware distribution in the master corpus")
print("=" * 64)
for passed in range(len(check_order_v2), -1, -1):
    print(f"  {passed}/{len(check_order_v2)}: {observed_score_counts_v2[passed]:,}")

print()
print("Tier transition table: original n/7 -> cleanliness-aware n/8")
print("=" * 64)
for (tier_v1, tier_v2), count in transition_counts_v2.most_common():
    print(f"  {tier_v1:28s} -> {tier_v2:28s} {count:,}")

print()
print("Poisson-binomial null fit for n/8")
print("=" * 64)
print(f"observed mean score                         : {mean_obs_v2:.4f}")
print(f"null mean score                             : {mean_null_v2:.4f}")
print(f"observed score variance                     : {var_obs_v2:.4f}")
print(f"null score variance                         : {var_null_v2:.4f}")
print(f"overdispersion ratio                        : {overdispersion_ratio_v2:.4f}")
print(f"Pearson chi-square statistic                : {chi2_stat_v2:.4f}")
print(f"Monte Carlo goodness-of-fit p               : {bootstrap_p_value_v2:.6f}")
print(f"95% bootstrap CI for overdispersion ratio   : [{overdispersion_ci_v2[0]:.4f}, {overdispersion_ci_v2[1]:.4f}]")

print()
print("Single-failure bottleneck analysis for the 7/8 shoulder")
print("=" * 64)
print(f"total 7/8 entries                           : {seven_of_eight_total:,}")
print(f"fail only non_mutation_suite_path           : {single_failure_counts_v2['non_mutation_suite_path']:,}")
print(f"share of 7/8 explained by mutation-only     : {mutation_only_share_v2:.4f}")
print(f"95% bootstrap CI for mutation-only share    : [{mutation_share_ci_v2[0]:.4f}, {mutation_share_ci_v2[1]:.4f}]")

print()
print("Exact single-failure counts by n/8 check")
print("=" * 64)
for cid in check_order_v2:
    print(f"  {cid:28s} {single_failure_counts_v2[cid]:,}")

print()
print("Phi coefficients against non_mutation_suite_path")
print("=" * 64)
for cid, phi in sorted(phi_against_mutation.items(), key=lambda kv: abs(kv[1]), reverse=True):
    print(f"  {cid:28s} {phi: .4f}")


## Statistical Analysis G — Direct Comparative Structure of `n/7` and `n/8`

The earlier `n/7` battery and the new `n/8` companion diagnostics answer different questions, but they are not yet a direct paired comparison because they are framed separately. This subsection closes that gap by comparing the two readiness views on the **same master-corpus sample**.

Two exact mathematical facts motivate the comparison:

1. If `S7` is the original seven-check score and `M` is the binary indicator for `non_mutation_suite_path`, then the cleanliness-aware score satisfies the identity `S8 = S7 + M`.
2. Because of that identity, the `n/8` histogram must be obtainable from the `n/7` histogram by splitting each `n/7` score bucket into mutation and non-mutation components and shifting only the clean component one step to the right.

This cell therefore reports three comparative diagnostics:

1. a paired benchmark-eligibility transition table between the `n/7` and `n/8` tier systems;
2. an exact histogram-decomposition check verifying the identity `N8(k) = N7_mut(k) + N7_clean(k-1)` on the master corpus; and
3. an excess-variance decomposition relative to the Poisson-binomial null, using the identity
   `excess_var_8 - excess_var_7 = 2 * sum_i Cov(X_i, M)`
   where `X_i` are the original seven benchmark checks and `M` is the cleanliness indicator.

The third diagnostic is especially useful because it explains *why* the `n/8` profile can become underdispersed even though the original `n/7` profile showed strong structured deviation from its own null. If the added cleanliness criterion has sufficiently negative covariance with the original checks, it can more than offset the positive dependence structure that drove the `n/7` shoulder.

**Principal variables introduced in the code cell**

- `scores7_master` and `scores8_master`: paired `n/7` and `n/8` score vectors on the same master-corpus rows
- `eligible7` and `eligible8`: paired benchmark-eligibility indicators under the original and cleanliness-aware tier systems
- `obs7_master`, `obs8_master`, `mut_hist7`, and `clean_hist7`: observed histograms used in the exact shift decomposition
- `excess_var7_master` and `excess_var8_master`: deviations of the two observed score variances from their respective Poisson-binomial null variances
- `cov_with_mutation` and `added_covariance_term`: covariance summary showing how the eighth criterion changes the null-gap algebraically


In [ ]:
from pathlib import Path
import json
from collections import Counter

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
phase3_master = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_master_processable_enriched.jsonl"

check_order7_master = [
    "validated_execution",
    "high_extraction_confidence",
    "no_demo_scaffolding",
    "no_cleanup_candidate",
    "minimum_code_lines",
    "minimum_gate_count",
    "trusted_retrieval_strategy",
]

rows7 = []
mutation_indicator = []
scores7_master = []
scores8_master = []
tiers7 = []
tiers8 = []

with open(phase3_master, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        meta = json.loads(line).get("metadata", {})
        passed7 = set(meta.get("benchmark_passed_checks") or [])
        row7 = [1 if cid in passed7 else 0 for cid in check_order7_master]
        rows7.append(row7)

        mut_flag = 0 if meta.get("mutation_suite_candidate") is True else 1
        mutation_indicator.append(mut_flag)

        s7 = meta.get("benchmark_checks_passed")
        if s7 is None:
            s7 = int(sum(row7))
        s8 = meta.get("benchmark_checks_passed_v2")
        if s8 is None:
            s8 = s7 + mut_flag

        scores7_master.append(int(s7))
        scores8_master.append(int(s8))
        tiers7.append(meta.get("benchmark_suitability_tier", "<missing>"))
        tiers8.append(meta.get("benchmark_suitability_tier_v2", "<missing>"))

X7_master = np.asarray(rows7, dtype=np.int8)
M_master = np.asarray(mutation_indicator, dtype=np.int8)
scores7_master = np.asarray(scores7_master, dtype=np.int16)
scores8_master = np.asarray(scores8_master, dtype=np.int16)

eligible7 = np.asarray([1 if t in {"strict_core_candidate", "extended_core_candidate"} else 0 for t in tiers7], dtype=np.int8)
eligible8 = np.asarray([1 if t in {"strict_core_candidate", "extended_core_candidate"} else 0 for t in tiers8], dtype=np.int8)

both_eligible = int(np.sum((eligible7 == 1) & (eligible8 == 1)))
only_7 = int(np.sum((eligible7 == 1) & (eligible8 == 0)))
only_8 = int(np.sum((eligible7 == 0) & (eligible8 == 1)))
neither = int(np.sum((eligible7 == 0) & (eligible8 == 0)))

obs7_master = np.bincount(scores7_master, minlength=8)
obs8_master = np.bincount(scores8_master, minlength=9)
mut_hist7 = np.bincount(scores7_master[M_master == 0], minlength=8)
clean_hist7 = np.bincount(scores7_master[M_master == 1], minlength=8)
expected_from_shift = np.zeros(9, dtype=int)
for k in range(9):
    same_bin_mut = mut_hist7[k] if k < len(mut_hist7) else 0
    shifted_clean = clean_hist7[k - 1] if 0 <= k - 1 < len(clean_hist7) else 0
    expected_from_shift[k] = int(same_bin_mut + shifted_clean)

shift_identity_holds = np.array_equal(obs8_master, expected_from_shift)

pass_rates7_master = X7_master.mean(axis=0)
pass_rates8_master = np.concatenate([pass_rates7_master, [M_master.mean()]])
var_obs7_master = float(scores7_master.var(ddof=0))
var_obs8_master = float(scores8_master.var(ddof=0))
var_null7_master = float(np.sum(pass_rates7_master * (1.0 - pass_rates7_master)))
var_null8_master = float(np.sum(pass_rates8_master * (1.0 - pass_rates8_master)))
excess_var7_master = var_obs7_master - var_null7_master
excess_var8_master = var_obs8_master - var_null8_master
cov_with_mutation = np.array([float(np.cov(X7_master[:, i], M_master, ddof=0)[0, 1]) for i in range(X7_master.shape[1])])
added_covariance_term = float(2.0 * np.sum(cov_with_mutation))
identity_gap = float((excess_var8_master - excess_var7_master) - added_covariance_term)

def fmt_pct(num, den):
    return 100.0 * num / den if den else float("nan")

print("Paired benchmark-eligibility transition table on the master corpus")
print("=" * 72)
print(f"eligible in both n/7 and n/8                 : {both_eligible:,} ({fmt_pct(both_eligible, len(scores7_master)):.1f}%)")
print(f"eligible only in n/7                         : {only_7:,} ({fmt_pct(only_7, len(scores7_master)):.1f}%)")
print(f"eligible only in n/8                         : {only_8:,} ({fmt_pct(only_8, len(scores7_master)):.1f}%)")
print(f"not eligible in either view                 : {neither:,} ({fmt_pct(neither, len(scores7_master)):.1f}%)")
print(f"retention rate among n/7-eligible entries   : {fmt_pct(both_eligible, both_eligible + only_7):.4f}%")

print()
print("Exact histogram shift identity from n/7 to n/8")
print("=" * 72)
print(f"identity N8(k) = N7_mut(k) + N7_clean(k-1) holds exactly: {shift_identity_holds}")
for k in range(8, -1, -1):
    observed = int(obs8_master[k])
    explained = int(expected_from_shift[k])
    same_bin_mut = int(mut_hist7[k]) if k < len(mut_hist7) else 0
    shifted_clean = int(clean_hist7[k - 1]) if 0 <= k - 1 < len(clean_hist7) else 0
    print(
        f"  {k}/8: observed={observed:>6,}  from mutation-same-bin={same_bin_mut:>6,}  "
        f"from clean-shift={shifted_clean:>6,}  total={explained:>6,}"
    )

print()
print("Excess-variance decomposition relative to the Poisson-binomial null")
print("=" * 72)
print(f"n/7 observed variance                        : {var_obs7_master:.4f}")
print(f"n/7 null variance                            : {var_null7_master:.4f}")
print(f"n/7 excess variance                          : {excess_var7_master:.4f}")
print(f"n/8 observed variance                        : {var_obs8_master:.4f}")
print(f"n/8 null variance                            : {var_null8_master:.4f}")
print(f"n/8 excess variance                          : {excess_var8_master:.4f}")
print(f"change in excess variance                    : {excess_var8_master - excess_var7_master:.4f}")
print(f"2 * sum_i Cov(X_i, non_mutation_suite_path)  : {added_covariance_term:.4f}")
print(f"identity gap (should be near 0)              : {identity_gap:.8f}")

print()
print("Covariance of the added cleanliness criterion with each original check")
print("=" * 72)
for cid, cov in sorted(zip(check_order7_master, cov_with_mutation), key=lambda kv: abs(kv[1]), reverse=True):
    print(f"  {cid:28s} {cov: .6f}")


## Phase 4 — Late-Stage Benchmark Recovery and Rebalancing

This phase is intentionally placed after master-corpus construction and pre-seed enrichment because it does not define the working input to seed generation or paraphrasing. Instead, it evaluates end-stage benchmark and release-manifest candidates that can be reconstructed from the fully processed corpus once the later instruction-generation pipeline has been completed.

This late-stage position also motivates the introduction of a second benchmark-readiness view. The original `n/7` score is retained as the historical and analytically continuous benchmark-readiness metric because all earlier Phase 3 counts, strict/extended tier definitions, and statistical tests were computed on that seven-check profile. It answers the question: how strong is this circuit under the original execution, extraction-quality, structural-threshold, and provenance criteria?

The proposed `n/8` view does not replace that score. Instead, it adds one explicit cleanliness criterion, `non_mutation_suite_path`, so users can distinguish intrinsically strong benchmark candidates from mutation-suite or bug-stress entries without losing visibility into the original seven-check readiness structure. In practical terms:

1. `n/7` remains the continuity score for historical comparison and analytical reproducibility.
2. `n/8` becomes the cleanliness-aware packaging score for late-stage benchmark and release filtering.
3. Mutation-derived entries can still be recognized as structurally strong stress examples rather than being silently conflated with the clean benchmark subset.

The three cells in this phase therefore remain useful as auditable end-stage filters: repository-cap analysis on the cleaned extended benchmark, materialization of capped candidate manifests for comparison, and threshold-relaxation audits on the cleaned Tier 2 reserve. They should be read as release and benchmark analyses, not as the primary processing path.


### Phase 4A — Dual Readiness Audit and Repository-Cap Recovery

This cell begins by printing both benchmark-readiness views on the cleaned extended benchmark audit set. The `n/7` profile preserves continuity with the original benchmark analysis, while the `n/8` profile adds the mutation-suite cleanliness criterion so that the effect of contamination control is visible numerically before any capped manifests are compared.

After that dual-score audit, the cell evaluates a sequence of repository-cap policies on the cleaned extended benchmark audit view. The purpose is to identify a balanced recovery regime that increases usable benchmark size while preventing `QDP-FSL` or any other repository from dominating the resulting late-stage development or release pool. The cell also reports the corresponding license mix for each cap level so that size and governance can be assessed together.

**Principal variables used in the code cell**

- `score_counts_v1` and `score_counts_v2`: score distributions under the original and cleanliness-aware readiness profiles on the cleaned extended audit set
- `cap_values`: candidate repository caps evaluated on the cleaned extended benchmark
- `cap_rows`: tabular recovery summary containing total size, dominant-repository share, and license composition for each cap scenario
- `extended_entries`: in-memory representation of the cleaned extended benchmark used to simulate capped manifests


In [ ]:
from pathlib import Path
import json
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PQID").exists())
cleaned_extended_core = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_cleaned_enriched.jsonl"

cap_values = [10, 15, 20, 25, 50]
extended_entries = []
score_counts_v1 = Counter()
score_counts_v2 = Counter()

with open(cleaned_extended_core, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            obj = json.loads(line)
            meta = obj.get("metadata", {})
            repo = f"{meta.get('repo_owner')}/{meta.get('repo_name')}"
            score_v1 = meta.get("benchmark_checks_passed")
            score_v2 = meta.get("benchmark_checks_passed_v2")
            if score_v2 is None and score_v1 is not None:
                score_v2 = score_v1 + (1 if meta.get("mutation_suite_candidate") is not True else 0)
            if score_v1 is not None:
                score_counts_v1[score_v1] += 1
            if score_v2 is not None:
                score_counts_v2[score_v2] += 1
            extended_entries.append((repo, obj))

print("Dual readiness audit on cleaned extended benchmark")
print("=" * 52)
print("n/7 distribution:")
for passed in sorted(score_counts_v1, reverse=True):
    print(f"  {passed}/7: {score_counts_v1[passed]:,}")
print()
print("n/8 distribution:")
for passed in sorted(score_counts_v2, reverse=True):
    print(f"  {passed}/8: {score_counts_v2[passed]:,}")
print()

cap_rows = []
for cap in cap_values:
    seen = Counter()
    kept = []
    for repo, obj in extended_entries:
        if seen[repo] < cap:
            seen[repo] += 1
            kept.append((repo, obj))
    repo_counts = Counter(repo for repo, _ in kept)
    license_counts = Counter()
    total = len(kept)
    for _, obj in kept:
        meta = obj.get("metadata", {})
        license_counts[meta.get("license_category") or "<missing>"] += 1
    top_repo, top_count = repo_counts.most_common(1)[0]
    cap_rows.append({
        "cap": cap,
        "total": total,
        "unique_repos": len(repo_counts),
        "top_repo": top_repo,
        "top_share": top_count / total if total else 0.0,
        "permissive": license_counts.get("permissive", 0),
        "copyleft": license_counts.get("copyleft", 0),
        "no_license": license_counts.get("no_license", 0),
    })

print("Repository-cap recovery audit")
print("=" * 40)
for row in cap_rows:
    print(
        f"cap={row['cap']:>2} | total={row['total']:,} | unique_repos={row['unique_repos']:,} | "
        f"top_repo={row['top_repo']} ({100 * row['top_share']:.1f}%) | "
        f"permissive={row['permissive']:,} | copyleft={row['copyleft']:,} | no_license={row['no_license']:,}"
    )


### Phase 4B — Balanced Extended Export Generation

This cell materializes two concrete `balanced_extended` candidate manifests from the cleaned extended benchmark audit view: a `cap20` version and a `cap25` version. These are not the default working corpus for instruction generation. They are written here so that late-stage release and benchmark comparisons can be based on explicit, inspectable files rather than on informal calculations.

**Principal variables used in the code cell**

- `balanced_cap20_path` and `balanced_cap25_path`: candidate balanced extended manifests
- `write_capped_subset()`: helper that writes a repository-capped subset while preserving record order


In [ ]:
from collections import Counter
import json
from pathlib import Path

balanced_cap20_path = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_balanced_cap20_enriched.jsonl"
balanced_cap25_path = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_core_extended_balanced_cap25_enriched.jsonl"

def write_capped_subset(entries, cap, output_path):
    seen = Counter()
    tmp_path = Path(str(output_path) + ".tmp")
    kept = 0
    with open(tmp_path, "w", encoding="utf-8") as out:
        for repo, obj in entries:
            if seen[repo] >= cap:
                continue
            seen[repo] += 1
            kept += 1
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
    tmp_path.replace(output_path)
    return kept

cap20_count = write_capped_subset(extended_entries, 20, balanced_cap20_path)
cap25_count = write_capped_subset(extended_entries, 25, balanced_cap25_path)

print("balanced extended exports written")
print(f"  cap20: {cap20_count:,} -> {balanced_cap20_path.name}")
print(f"  cap25: {cap25_count:,} -> {balanced_cap25_path.name}")


### Phase 4C — Threshold-Relaxation Recovery Audit

This cell audits how many additional non-mutation entries could be recovered from the cleaned Tier 2 reserve under mild rule relaxations. The audit keeps the existing high-confidence, no-demo, no-cleanup requirements and changes only the structural thresholds. The purpose is to measure the size of the near-core reserve without silently changing the benchmark definition or the main working corpus.

**Principal variables used in the code cell**

- `relaxation_profiles`: alternative threshold profiles evaluated against the cleaned Tier 2 reserve
- `relaxation_rows`: recovery summary for each profile, including promoted extended and promoted strict counts
- `mutation_suite_candidate`: metadata flag used to exclude mutation-suite entries from the recovery audit


In [ ]:
import json
from collections import Counter

cleaned_extended_tier2 = ROOT / "PQID/data/processed/circuits_unified_plus_phase2_plus_phase3_tier2_extended_cleaned.jsonl"
relaxation_profiles = [
    ("code4_gate2", 4, 2),
    ("code5_gate1", 5, 1),
    ("code4_gate1", 4, 1),
]

relaxation_rows = []
for label, min_code_lines, min_gate_count in relaxation_profiles:
    promoted_extended = 0
    promoted_strict = 0
    license_counts = Counter()
    with open(cleaned_extended_tier2, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            meta = obj.get("metadata", {})
            if meta.get("mutation_suite_candidate") is True:
                continue
            validated = meta.get("validation_status") == "validated"
            high_conf = meta.get("extraction_confidence") == "high"
            no_demo = meta.get("contains_demo_scaffolding") is not True
            no_cleanup = meta.get("cleanup_candidate") is not True
            code_ok = (meta.get("code_lines") or 0) >= min_code_lines
            gate_ok = (meta.get("gate_count") or 0) >= min_gate_count
            trusted = meta.get("retrieval_strategy") != "empirical_promoted_repo"
            if validated and high_conf and no_demo and no_cleanup and code_ok and gate_ok:
                promoted_extended += 1
                license_counts[meta.get("license_category") or "<missing>"] += 1
                if trusted:
                    promoted_strict += 1
    relaxation_rows.append({
        "profile": label,
        "promoted_extended": promoted_extended,
        "promoted_strict": promoted_strict,
        "permissive": license_counts.get("permissive", 0),
        "copyleft": license_counts.get("copyleft", 0),
        "no_license": license_counts.get("no_license", 0),
    })

print("Threshold-relaxation recovery audit")
print("=" * 40)
for row in relaxation_rows:
    print(
        f"{row['profile']}: promoted_extended={row['promoted_extended']:,} | "
        f"promoted_strict={row['promoted_strict']:,} | permissive={row['permissive']:,} | "
        f"copyleft={row['copyleft']:,} | no_license={row['no_license']:,}"
    )


## Phase 5 — Publication-Facing Artifact Names

The internal pipeline filenames remain intentionally verbose because they encode acquisition lineage and rebuild history. Names such as `circuits_unified_plus_phase2_plus_phase3_master_processable_enriched.jsonl` are useful during development because they preserve exact provenance: they tell us which acquisition stages were merged, whether the file is enriched, and how it sits relative to other intermediate artifacts.

That same property becomes a weakness in the final publication layer. For a public notebook, dataset card, manuscript, or release bundle, those lineage-heavy names are visually noisy and make the artifact set look less deliberate than it really is. They are also harder for reviewers and downstream users to scan quickly.

This phase therefore introduces a clean publication-facing alias layer at the very end of the workflow. The underlying internal files are not renamed or destroyed. Instead, short aliases such as `pqid_2026_master_corpus.jsonl` and `pqid_2026_benchmark_strict_clean.jsonl` are materialized from the current processed outputs.

This is an intentional two-layer design rather than a cosmetic rename:

1. the internal lineage layer preserves reproducibility and exact pipeline traceability;
2. the publication layer presents the same artifacts with stable, professional names suitable for release.

Because the alias layer is produced only after the full processing pipeline has run, reviewers can still reconstruct the provenance of every public artifact back to the original internal file that generated it. In other words, we do not trade auditability for presentation quality; we keep both.


### Phase 5A — Export Clean Publication Aliases

This cell materializes the clean publication-facing aliases in the processed-data directory. By default it uses hardlinks so large JSONL files are not duplicated unnecessarily on disk. If hardlink creation is unavailable, the script falls back to regular file copies.

The resulting aliases are the filenames that should be used in the final public notebook, Hugging Face dataset card, manuscript data-records section, and release bundle. The longer internal filenames remain the authoritative lineage-preserving names inside the working pipeline.

This separation serves three purposes simultaneously:

1. it keeps the development pipeline stable because no internal intermediate is renamed in place;
2. it keeps the final release professional because users see short, intentional artifact names;
3. it keeps the release auditable because each public alias is explicitly tied back to its internal source artifact.

**Principal artifacts created by the export script**

- `pqid_2026_raw_github_circuits.jsonl`
- `pqid_2026_enriched_github_circuits.jsonl`
- `pqid_2026_master_corpus.jsonl`
- `pqid_2026_benchmark_strict.jsonl`
- `pqid_2026_benchmark_extended.jsonl`
- `pqid_2026_benchmark_strict_clean.jsonl`
- `pqid_2026_benchmark_extended_clean.jsonl`
- corresponding clean markdown report names


In [ ]:
result = subprocess.run(
    [
        str(PYTHON_EXE),
        str(ROOT / "PQID/scripts/export_publication_artifacts.py"),
        "--mode",
        "hardlink",
        "--overwrite",
    ],
    check=True,
        capture_output=True,
        text=True,
)

if result.stdout.strip():

    print(result.stdout.strip())

if result.stderr.strip():

    print("stderr")

    print(result.stderr.strip())

print("publication-artifact export completed with return code:", result.returncode)